# **Segmentación y extracción de características de masas en mamografías**



Este notebook acompaña el informe del Trabajo Práctico Integrador de Procesamiento de Imágenes Biomédicas. El objetivo general del trabajo es implementar y evaluar un pipeline de procesamiento de imágenes para segmentar masas mamográficas a partir de regiones de interés y, posteriormente, extraer características cuantitativas de las máscaras obtenidas.

El flujo de trabajo parte de imágenes mamográficas en formato DICOM y de las máscaras de referencia provistas por la base de datos. A partir de esta información se construye una ventana local de análisis centrada en la lesión. Sobre esa ventana se aplica un preprocesamiento adaptativo orientado a mejorar el contraste local de la masa respecto del tejido circundante. Luego se comparan tres estrategias de segmentación: K-means, contorno activo geodésico morfológico y un método híbrido que combina ambos enfoques.

Finalmente, las segmentaciones obtenidas se comparan contra la máscara de referencia mediante métricas desempeño como Dice y Jaccard. Además, se calculan características geométricas simples de las máscaras segmentadas, con el fin de caracterizar cuantitativamente la región detectada.

El notebook fue organizado para que cada sección incluya: una breve justificación metodológica, las funciones principales utilizadas, visualizaciones representativas del procesamiento y resultados cuantitativos resumidos.


## Esquema general del pipeline



El procesamiento implementado sigue la siguiente secuencia:

→ Emparejamiento de archivos según metadatos  
→ Construcción de ventana local de análisis  
→ Normalización de intensidades  
→ Preprocesamiento adaptativo con transformación logarítmica y CLAHE  
→ Segmentación mediante K-means, MorphGAC e híbrido  
→ Correcciones morfológicas y control de bordes  
→ Comparación contra máscara de referencia  
→ Cálculo de métricas de segmentación  
→ Extracción de características de forma, borde e intensidad  
→ Análisis de resultados


# *1. Lectura de archivos y configuración inicial*

En esta primera sección se importan las librerías necesarias para el procesamiento, se definen las rutas de trabajo y se preparan las funciones iniciales para acceder a los archivos DICOM y CSV.

## Imports y Paths

Se importan las librerías necesarias para la lectura de imágenes DICOM, el manejo de tablas, el procesamiento de imágenes, la segmentación, el cálculo de métricas y la visualización de resultados.

También se definen las rutas de trabajo en Google Drive, incluyendo las carpetas de mamografías completas, ROIs, máscaras de referencia, archivo CSV y carpetas de salida.

In [ ]:
# ============================================================
# CELDA 1 - Imports y configuración de rutas
# ============================================================

# En Colab, si falta alguna librería:
!pip install pydicom scikit-image scikit-learn -q

from pathlib import Path
import re
import gc
import traceback
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pydicom
from pydicom.multival import MultiValue
from pydicom.pixel_data_handlers.util import apply_voi_lut

from scipy import ndimage as ndi

from skimage import filters, exposure, morphology, measure, segmentation, restoration
from skimage.segmentation import morphological_geodesic_active_contour, inverse_gaussian_gradient
from skimage.transform import resize
from sklearn.cluster import KMeans

from IPython.display import display

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["image.cmap"] = "gray"


# ============================================================
# Montaje de Drive
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("No se montó Google Drive. Si estás fuera de Colab, ignorar este aviso.")


# ============================================================
# Rutas principales
# Ajustar sólo si tu estructura de carpetas cambió.
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/TP-PSIB/TPI")

DICOM_DIR = BASE_DIR / "IMG" / "MG"
ROI_DIR   = BASE_DIR / "IMG" / "ROI"
MASK_DIR  = BASE_DIR / "IMG" / "MASK"

CSV_PATH = BASE_DIR / "mass_case_description_train_set.csv"

OUT_DIR = BASE_DIR / "resultados_pipeline_final"
FIG_DIR = OUT_DIR / "figuras_pipeline_final"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


print("BASE_DIR:", BASE_DIR, "existe:", BASE_DIR.exists())
print("DICOM_DIR:", DICOM_DIR, "existe:", DICOM_DIR.exists())
print("ROI_DIR:", ROI_DIR, "existe:", ROI_DIR.exists())
print("MASK_DIR:", MASK_DIR, "existe:", MASK_DIR.exists())
print("CSV_PATH:", CSV_PATH, "existe:", CSV_PATH.exists())
print("OUT_DIR:", OUT_DIR)


# ============================================================
# Parámetros generales del batch
# ============================================================

# Para probar primero, usar por ejemplo MAX_CASOS = 5.
# Para correr todo, poner MAX_CASOS = None.
MAX_CASOS = None
START_IDX = 0

# Parámetro principal del híbrido GAC + K-means.
RADIO_BANDA_HIBRIDO = 6

# Visualización y guardado para cuidar RAM.
MOSTRAR_PRIMEROS_N = 3
GUARDAR_FIGURAS_PRIMEROS_N = 8
LIMPIAR_RAM = True

# Umbrales para análisis de fallas.
UMBRAL_DICE_BAJO = 0.60
UMBRAL_SUBSEGMENTA = 0.70
UMBRAL_SOBRESEGMENTA = 1.50


## Inspección de DICOMS








Las imágenes se encuentran en formato DICOM, que no sólo almacena la matriz de píxeles, sino también metadatos asociados al estudio, como identificadores del paciente, lateralidad (LEFT o RIGHT) y vista mamográfica (CC o MLO).

En esta sección se leen los archivos correspondientes a la mamografía completa, la ROI y la máscara de referencia. A partir de los metadatos y de la información presente en las rutas, se construye una tabla que permite emparejar correctamente los tres archivos de cada lesión.

El objetivo de esta etapa es obtener una tabla local de casos completos, donde cada fila tenga asociadas las rutas de su mamografía, ROI y máscara.

In [ ]:
# ============================================================
# CELDA 2 - Leer DICOM y unir MG + ROI + MASK locales
# ============================================================

def listar_archivos(carpeta):
    return sorted([p for p in Path(carpeta).rglob("*") if p.is_file()])


def tag(ds, nombre):
    valor = getattr(ds, nombre, "")
    return "" if valor is None else str(valor)


def extraer_paciente(texto):
    m = re.search(r"P_\d+", str(texto).upper())
    return m.group(0) if m else np.nan


def extraer_lateralidad(texto):
    texto = str(texto).upper()
    if "LEFT" in texto:
        return "LEFT"
    if "RIGHT" in texto:
        return "RIGHT"
    return np.nan


def extraer_vista(texto):
    texto = str(texto).upper()
    if "MLO" in texto:
        return "MLO"
    if "CC" in texto:
        return "CC"
    return np.nan


def extraer_abnormality_id(texto):
    texto = str(texto).upper()
    m = re.search(r"P_\d+_(LEFT|RIGHT)_(CC|MLO)_(\d+)", texto)
    return str(int(m.group(3))) if m else np.nan


def leer_spacing(ds):
    for nombre in ["PixelSpacing", "ImagerPixelSpacing", "NominalScannedPixelSpacing"]:
        valor = getattr(ds, nombre, None)
        if valor is None:
            continue

        nums = re.findall(r"[-+]?\d*\.?\d+", str(valor))

        if len(nums) >= 2:
            return float(nums[0]), float(nums[1]), nombre

    return np.nan, np.nan, np.nan


def leer_dicom(ruta, origen):
    ds = pydicom.dcmread(str(ruta), stop_before_pixels=True, force=True)

    texto = " ".join([
        tag(ds, "PatientID"),
        tag(ds, "PatientName"),
        tag(ds, "ViewPosition"),
        tag(ds, "ImageLaterality"),
        tag(ds, "Laterality"),
        str(ruta)
    ])

    spacing_y, spacing_x, spacing_fuente = leer_spacing(ds)

    return {
        "origen": origen,
        "ruta": str(ruta),

        "patient_id": extraer_paciente(texto),
        "lateralidad": extraer_lateralidad(texto),
        "vista": extraer_vista(texto),
        "abnormality_id": extraer_abnormality_id(texto),

        "Rows": tag(ds, "Rows"),
        "Columns": tag(ds, "Columns"),
        "spacing_y_mm": spacing_y,
        "spacing_x_mm": spacing_x,
        "spacing_fuente": spacing_fuente
    }


# Leo todos los DICOM locales.
filas = []

for origen, carpeta in [("MG", DICOM_DIR), ("ROI", ROI_DIR), ("MASK", MASK_DIR)]:
    for ruta in listar_archivos(carpeta):
        try:
            filas.append(leer_dicom(ruta, origen))
        except Exception:
            pass

df_archivos = pd.DataFrame(filas)

# Claves internas:
# clave_mg = paciente + lado + vista
# clave_masa = paciente + lado + vista + número de masa
# El número de masa se conserva sólo como clave interna de unión.
df_archivos["clave_mg"] = (
    df_archivos["patient_id"].astype(str) + "_" +
    df_archivos["lateralidad"].astype(str) + "_" +
    df_archivos["vista"].astype(str)
)

df_archivos["clave_masa"] = (
    df_archivos["clave_mg"] + "_" +
    df_archivos["abnormality_id"].astype(str)
)


# Separo por tipo de archivo.
df_mg = df_archivos[df_archivos["origen"] == "MG"].copy()
df_roi = df_archivos[df_archivos["origen"] == "ROI"].copy()
df_mask = df_archivos[df_archivos["origen"] == "MASK"].copy()


# Preparo tablas chicas para unir.
mg = df_mg.drop_duplicates("clave_mg")[[
    "clave_mg", "ruta", "Rows", "Columns",
    "spacing_y_mm", "spacing_x_mm", "spacing_fuente"
]].rename(columns={
    "ruta": "ruta_mg",
    "Rows": "Rows_mg",
    "Columns": "Columns_mg"
})

roi = df_roi.drop_duplicates("clave_masa")[[
    "clave_masa", "ruta"
]].rename(columns={
    "ruta": "ruta_roi"
})

mask = df_mask.drop_duplicates("clave_masa")[[
    "clave_mg", "clave_masa",
    "patient_id", "lateralidad", "vista", "abnormality_id",
    "ruta"
]].rename(columns={
    "ruta": "ruta_mask"
})


# Esta es la unión local: una fila por masa/máscara local.
df_casos_local = (
    mask
    .merge(roi, on="clave_masa", how="left")
    .merge(mg, on="clave_mg", how="left")
)

df_casos_local["union_local_ok"] = (
    df_casos_local["ruta_mg"].notna() &
    df_casos_local["ruta_roi"].notna() &
    df_casos_local["ruta_mask"].notna()
)

# ============================================================
# Resumen y guardado de tablas de control
# ============================================================

print("DICOM leídos por tipo:")
display(
    df_archivos["origen"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "tipo_archivo", "origen": "cantidad"})
)

print("Masas/máscaras locales:", len(df_casos_local))
print("Uniones locales completas MG + ROI + MASK:", df_casos_local["union_local_ok"].sum())

print("\nResumen de uniones locales:")
display(
    df_casos_local[[
        "patient_id", "lateralidad", "vista", "abnormality_id", "union_local_ok"
    ]]
    .reset_index(drop=True)
    .head(10)
)

# Guardamos las tablas completas
df_archivos.to_csv(OUT_DIR / "inventario_dicom.csv", index=False)
df_casos_local.to_csv(OUT_DIR / "casos_locales_mg_roi_mask.csv", index=False)

print("\nTablas completas guardadas en:")
print(OUT_DIR / "inventario_dicom.csv")
print(OUT_DIR / "casos_locales_mg_roi_mask.csv")


## Extracción de características del CSV

Se agrega la información descriptiva de cada lesión proveniente del CSV de CBIS-DDSM, como densidad mamaria, forma de la masa, márgenes y diagnóstico.

Esta tabla se utilizará más adelante para interpretar los resultados de segmentación y extracción de características según las propiedades de cada caso.

In [ ]:
# ============================================================
# CELDA 3 - Agregar datos del CSV a los casos locales
# ============================================================

df_csv = pd.read_csv(CSV_PATH)

# Normalizo las columnas del CSV para que coincidan con lo extraído de los DICOM.
df_csv["patient_id_norm"] = df_csv["patient_id"].astype(str)
df_csv["lateralidad_norm"] = df_csv["left or right breast"].astype(str)
df_csv["vista_norm"] = df_csv["image view"].astype(str)
df_csv["abnormality_id_norm"] = df_csv["abnormality id"].apply(lambda x: str(int(x)))

df_csv["clave_mg"] = (
    df_csv["patient_id_norm"] + "_" +
    df_csv["lateralidad_norm"] + "_" +
    df_csv["vista_norm"]
)

df_csv["clave_masa"] = (
    df_csv["clave_mg"] + "_" +
    df_csv["abnormality_id_norm"]
)


# Me quedo sólo con las columnas del CSV que sirven para el análisis.
datos_csv = df_csv[[
    "clave_masa",
    "breast_density",
    "abnormality type",
    "mass shape",
    "mass margins",
    "assessment",
    "pathology",
    "subtlety"
]].drop_duplicates("clave_masa")


# Agrego esos datos a los casos locales.
df_casos = df_casos_local.merge(datos_csv, on="clave_masa", how="left")

df_casos = df_casos.rename(columns={
    "abnormality type": "abnormality_type",
    "mass shape": "mass_shape",
    "mass margins": "mass_margins"
})

df_casos["csv_ok"] = df_casos["breast_density"].notna()

# Identificador legible para tablas y figuras. El abnormality_id se mantiene
# sólo como clave interna para unir correctamente ROI, MASK y CSV.
df_casos["case_id"] = (
    df_casos["patient_id"].astype(str) + "_" +
    df_casos["lateralidad"].astype(str) + "_" +
    df_casos["vista"].astype(str)
)

df_casos_ok = df_casos[
    df_casos["union_local_ok"] &
    df_casos["csv_ok"]
].copy()


if MAX_CASOS is not None:
    df_casos_batch = df_casos_ok.iloc[START_IDX:START_IDX + MAX_CASOS].copy()
else:
    df_casos_batch = df_casos_ok.iloc[START_IDX:].copy()


# ============================================================
# Resumen y guardado de la tabla final de casos
# ============================================================

print("Casos locales:", len(df_casos))
print("Casos locales encontrados en el CSV:", df_casos["csv_ok"].sum())
print("Casos completos para procesar:", len(df_casos_ok))
print("Casos en el batch actual:", len(df_casos_batch))

print("\nCasos que se van a procesar, resumen:")
display(
    df_casos_batch[[
        "case_id",
        "breast_density",
        "mass_shape",
        "mass_margins",
        "assessment",
        "pathology",
        "subtlety"
    ]]
    .reset_index(drop=True)
)

# Guardo la tabla completa, incluyendo rutas, para trazabilidad.
df_casos.to_csv(OUT_DIR / "casos_con_csv.csv", index=False)
df_casos_batch.to_csv(OUT_DIR / "casos_batch_procesados.csv", index=False)

print("\nTablas completas guardadas en:")
print(OUT_DIR / "casos_con_csv.csv")
print(OUT_DIR / "casos_batch_procesados.csv")

# Sólo muestra una mini-revisión si algún caso local no encontró fila en el CSV.
if df_casos["csv_ok"].sum() < len(df_casos):
    print("\nCasos locales sin coincidencia en CSV, máximo 10:")
    display(
        df_casos[~df_casos["csv_ok"]][[
            "case_id", "patient_id", "lateralidad", "vista", "clave_masa"
        ]]
        .head(10)
        .reset_index(drop=True)
    )


# *2. Preprocesamiento*

## Lectura de imágenes y construcción de la ventana de análisis

Esta celda define las funciones necesarias para leer la mamografía completa, la ROI y la máscara de referencia de cada caso.

La función `leer_mamografia` permite obtener la imagen mamográfica desde el archivo DICOM. La función `leer_dicom_imagen` se utiliza para leer la ROI y la máscara. Luego, `redimensionar_mask_a_mg` ajusta la máscara al tamaño de la mamografía completa cuando hay pequeñas diferencias de dimensiones.

A partir de la máscara de referencia se calcula la posición de la lesión en la mamografía completa. Luego, con `calcular_ventana_centrada`, se recorta una ventana local centrada en la masa, usando como tamaño de referencia la ROI provista por la base. Finalmente, `normalizar_ventana_local` lleva las intensidades de la ventana a un rango comparable entre casos.


In [ ]:
# ============================================================
# CELDA 4 - Lectura, normalización y obtención de ventana
# ============================================================

def leer_mamografia(ruta_mg):
    ds = pydicom.dcmread(str(ruta_mg), force=True)
    mg = ds.pixel_array.astype(np.float32)

    try:
        mg = apply_voi_lut(mg, ds).astype(np.float32)
    except Exception:
        pass

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        mg = mg.max() - mg

    return mg


def leer_dicom_imagen(ruta):
    ds = pydicom.dcmread(str(ruta), force=True)
    img = ds.pixel_array.astype(np.float32)

    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        img = img.max() - img

    return img


def redimensionar_mask_a_mg(mask_raw, shape_mg):
    mask_bin = np.squeeze(mask_raw) > 0

    if mask_bin.shape == shape_mg:
        return mask_bin.astype(bool)

    mask_bin = resize(
        mask_bin.astype(np.uint8),
        output_shape=shape_mg,
        order=0,
        preserve_range=True,
        anti_aliasing=False
    ) > 0

    return mask_bin.astype(bool)


def calcular_ventana_centrada(cy, cx, alto, ancho, shape_img):
    H, W = shape_img

    alto = min(int(alto), H)
    ancho = min(int(ancho), W)

    y0 = int(cy) - alto // 2
    x0 = int(cx) - ancho // 2

    y0 = max(0, min(y0, H - alto))
    x0 = max(0, min(x0, W - ancho))

    y1 = y0 + alto
    x1 = x0 + ancho

    return y0, y1, x0, x1


def normalizar_ventana_local(ventana):
    lo, hi = np.percentile(ventana[np.isfinite(ventana)], [1, 99])

    ventana = (ventana - lo) / (hi - lo + 1e-8)
    ventana = np.clip(ventana, 0, 1).astype(np.float32)

    return ventana


def preparar_ventana_y_gt(caso):
    # Leo la mamografía completa sin normalizar. La normalización se aplica sólo sobre la ventana.
    mg = leer_mamografia(caso["ruta_mg"])

    # Leo máscara de referencia y ROI asociada a la lesión.
    mask_raw = leer_dicom_imagen(caso["ruta_mask"])
    roi_raw = leer_dicom_imagen(caso["ruta_roi"])

    # La máscara debe estar en coordenadas de la mamografía completa.
    mask_full = redimensionar_mask_a_mg(mask_raw, mg.shape)

    # Uso el tamaño de la ROI como tamaño de ventana de análisis.
    roi_2d = np.squeeze(roi_raw)
    alto_roi, ancho_roi = roi_2d.shape

    # Centro de la lesión según la máscara de referencia.
    y, x = np.where(mask_full)

    if len(y) == 0:
        raise ValueError("La máscara de lesión está vacía.")

    cy = int((y.min() + y.max()) / 2)
    cx = int((x.min() + x.max()) / 2)

    # Ventana centrada en la lesión, recortada directamente desde la mamografía.
    wy0, wy1, wx0, wx1 = calcular_ventana_centrada(
        cy, cx,
        alto_roi, ancho_roi,
        mg.shape
    )

    ventana = mg[wy0:wy1, wx0:wx1]
    gt_ventana = mask_full[wy0:wy1, wx0:wx1]

    # Normalización local de la ventana para que el procesamiento posterior
    # no dependa del rango original de intensidades de cada DICOM.
    ventana = normalizar_ventana_local(ventana)

    info = {
        "shape_mg": mg.shape,
        "shape_roi": roi_2d.shape,
        "shape_ventana": ventana.shape,
        "ventana_full": (wy0, wy1, wx0, wx1)
    }

    return ventana, gt_ventana.astype(bool), info


### Visualización de la construcción de la ventana

La siguientes figuras muestran cómo se construye la ventana de análisis. Esta visualización permite verificar que la ventana queda correctamente centrada en la lesión y que la máscara de referencia queda alineada con el recorte utilizado por el pipeline.


In [ ]:
# @title
# ============================================================
# Visualización 1 - Mamografía, máscara y superposición
# ============================================================

# Casos a visualizar.
# Elegir más o menos según lo que quieran mostrar.
IDX_VIS_BASE = [0, 1, 2, 3, 4, 5]
IDX_VIS_BASE = [i for i in IDX_VIS_BASE if i < len(df_casos_batch)]

# Color de superposición: fucsia visible sobre escala de grises
COLOR_OVERLAY_RGB = np.array([1.0, 0.0, 0.55])
ALPHA_MASK = 0.55


def crear_overlay_color(imagen_gray, mascara_bool, color_rgb=COLOR_OVERLAY_RGB, alpha=ALPHA_MASK):

    img = imagen_gray.astype(float)

    # Normalizo sólo para visualización.
    vmin, vmax = np.percentile(img[np.isfinite(img)], [1, 99])
    img_norm = np.clip((img - vmin) / (vmax - vmin + 1e-12), 0, 1)

    overlay = np.dstack([img_norm, img_norm, img_norm])

    mask = mascara_bool.astype(bool)
    overlay[mask] = (1 - alpha) * overlay[mask] + alpha * color_rgb

    return overlay


for idx in IDX_VIS_BASE:

    caso = df_casos_batch.iloc[idx]

    mg = leer_mamografia(caso["ruta_mg"])
    mask_raw = leer_dicom_imagen(caso["ruta_mask"])
    mask_full = redimensionar_mask_a_mg(mask_raw, mg.shape).astype(bool)

    vals_mg = mg[np.isfinite(mg)]
    vmin_mg, vmax_mg = np.percentile(vals_mg, [1, 99])

    overlay = crear_overlay_color(mg, mask_full)

    fig, axes = plt.subplots(
        1, 3,
        figsize=(15, 5),
        constrained_layout=True
    )

    fig.suptitle(
        f"Imágenes provistas por la base — {caso['patient_id']} | {caso['lateralidad']} | {caso['vista']}",
        fontsize=14
    )

    # 1. Mamografía completa
    axes[0].imshow(mg, cmap="gray", vmin=vmin_mg, vmax=vmax_mg)
    axes[0].set_title("1. Mamografía completa", fontsize=11, pad=10)
    axes[0].axis("off")

    # 2. Máscara completa
    axes[1].imshow(mask_full, cmap="gray")
    axes[1].set_title("2. Máscara binaria completa", fontsize=11, pad=10)
    axes[1].axis("off")

    # 3. Superposición
    axes[2].imshow(overlay)
    axes[2].contour(mask_full, levels=[0.5], colors=["white"], linewidths=0.7)
    axes[2].set_title("3. Superposición mamografía + máscara", fontsize=11, pad=10)
    axes[2].axis("off")

    plt.savefig(
        FIG_DIR / f"base_mg_mask_overlay_{idx:03d}_{caso['case_id']}.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# @title
# ============================================================
# Visualización 2 - Construcción de la ventana local de análisis
# ============================================================

from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# Casos a visualizar
IDX_VIS_VENTANA = [0, 1, 2, 3, 4, 5]
IDX_VIS_VENTANA = [i for i in IDX_VIS_VENTANA if i < len(df_casos_batch)]

# Colores suaves
COLOR_BBOX = "#D8A7A7"
COLOR_VENTANA = "#8FBBD9"
COLOR_CENTRO = "#D6B65F"
COLOR_GT = "#B07A8C"


# ============================================================
# Leyenda única
# ============================================================

handles = [
    Rectangle(
        (0, 0), 1, 1,
        fill=False,
        edgecolor=COLOR_BBOX,
        linewidth=2,
        label="Bounding box de la máscara"
    ),
    Rectangle(
        (0, 0), 1, 1,
        fill=False,
        edgecolor=COLOR_VENTANA,
        linewidth=2,
        label="Ventana final"
    ),
    Line2D(
        [0], [0],
        marker="+",
        color=COLOR_CENTRO,
        linestyle="None",
        markersize=11,
        markeredgewidth=2,
        label="Centro calculado"
    ),
    Line2D(
        [0], [0],
        color=COLOR_GT,
        linewidth=2,
        label="Máscara de referencia"
    )
]

fig_leg = plt.figure(figsize=(10, 0.8))
fig_leg.legend(
    handles=handles,
    loc="center",
    ncol=4,
    frameon=True,
    fontsize=9
)
plt.axis("off")

plt.savefig(
    FIG_DIR / "leyenda_construccion_ventana.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# Figuras por caso
# ============================================================

for idx in IDX_VIS_VENTANA:

    caso = df_casos_batch.iloc[idx]

    mg = leer_mamografia(caso["ruta_mg"])
    roi_raw = leer_dicom_imagen(caso["ruta_roi"])
    mask_raw = leer_dicom_imagen(caso["ruta_mask"])

    roi_2d = np.squeeze(roi_raw)
    mask_full = redimensionar_mask_a_mg(mask_raw, mg.shape).astype(bool)

    y, x = np.where(mask_full)

    if len(y) == 0:
        raise ValueError(f"La máscara de referencia está vacía para el caso {idx}.")

    y_min, y_max = int(y.min()), int(y.max())
    x_min, x_max = int(x.min()), int(x.max())

    cy = int((y_min + y_max) / 2)
    cx = int((x_min + x_max) / 2)

    alto_roi, ancho_roi = roi_2d.shape

    wy0, wy1, wx0, wx1 = calcular_ventana_centrada(
        cy,
        cx,
        alto_roi,
        ancho_roi,
        mg.shape
    )

    ventana_raw = mg[wy0:wy1, wx0:wx1]
    ventana_norm = normalizar_ventana_local(ventana_raw)
    mask_ventana = mask_full[wy0:wy1, wx0:wx1]

    vals_mg = mg[np.isfinite(mg)]
    vmin_mg, vmax_mg = np.percentile(vals_mg, [1, 99])

    vals_roi = roi_2d[np.isfinite(roi_2d)]
    vmin_roi, vmax_roi = np.percentile(vals_roi, [1, 99])

    vals_win = ventana_raw[np.isfinite(ventana_raw)]
    vmin_win, vmax_win = np.percentile(vals_win, [1, 99])

    pad_y = max(25, int(0.35 * (wy1 - wy0)))
    pad_x = max(25, int(0.35 * (wx1 - wx0)))

    zy0 = max(0, wy0 - pad_y)
    zy1 = min(mg.shape[0], wy1 + pad_y)
    zx0 = max(0, wx0 - pad_x)
    zx1 = min(mg.shape[1], wx1 + pad_x)

    fig, axes = plt.subplots(
        1, 4,
        figsize=(18, 5),
        constrained_layout=True
    )

    fig.suptitle(
        f"Construcción de ventana local — {caso['patient_id']} | {caso['lateralidad']} | {caso['vista']}",
        fontsize=14
    )

    axes[0].imshow(roi_2d, cmap="gray", vmin=vmin_roi, vmax=vmax_roi)
    axes[0].set_title(f"1. ROI provista\n{alto_roi} × {ancho_roi} px", fontsize=11, pad=10)
    axes[0].axis("off")

    axes[1].imshow(mg, cmap="gray", vmin=vmin_mg, vmax=vmax_mg)

    axes[1].add_patch(
        Rectangle(
            (x_min, y_min),
            x_max - x_min,
            y_max - y_min,
            fill=False,
            edgecolor=COLOR_BBOX,
            linewidth=2.2
        )
    )

    axes[1].add_patch(
        Rectangle(
            (wx0, wy0),
            wx1 - wx0,
            wy1 - wy0,
            fill=False,
            edgecolor=COLOR_VENTANA,
            linewidth=2.2
        )
    )

    axes[1].scatter(
        cx,
        cy,
        marker="+",
        s=150,
        color=COLOR_CENTRO,
        linewidths=2.2
    )

    axes[1].set_xlim(zx0, zx1)
    axes[1].set_ylim(zy1, zy0)
    axes[1].set_title("2. Localización de la ventana", fontsize=11, pad=10)
    axes[1].axis("off")

    axes[2].imshow(
        ventana_raw,
        cmap="gray",
        vmin=vmin_win,
        vmax=vmax_win
    )
    axes[2].set_title("3. Ventana recortada", fontsize=11, pad=10)
    axes[2].axis("off")

    axes[3].imshow(ventana_norm, cmap="gray", vmin=0, vmax=1)
    axes[3].contour(
        mask_ventana,
        levels=[0.5],
        colors=[COLOR_GT],
        linewidths=2
    )
    axes[3].set_title("4. Ventana normalizada + máscara", fontsize=11, pad=10)
    axes[3].axis("off")

    plt.savefig(
        FIG_DIR / f"construccion_ventana_{idx:03d}_{caso['case_id']}.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

## Preprocesamiento adaptativo sobre las ventanas


Esta celda define el preprocesamiento que se aplica sobre la ventana local antes de segmentar la masa. El objetivo es mejorar el contraste de la lesión respecto del tejido circundante a partir de un preprocesamiento que se adapta a las diferencias entre las masas, ya que pueden tener distinto contraste, densidad y relación con el fondo.

La función `mini_rois` construye una mini-ROI central y cuatro mini-ROIs periféricas. La mini-ROI central representa una aproximación de la zona donde se espera encontrar la masa. Las mini-ROIs periféricas se usan como referencia del fondo local.

`elegir_parametros_miniroi` calcula indicadores a partir de esas regiones:

* `separacion = (c50 - f50) / (f75 - f25)`: compara la mediana del centro con la mediana del fondo, normalizada por la dispersión del fondo. Si es alta, la masa se diferencia bien del fondo.
* `competencia_fondo = mean(fondo > c50)`: mide qué proporción del fondo tiene intensidades mayores que la mediana de la masa. Si es alta, el fondo compite con la masa y existe más riesgo de sobresegmentar tejido sano.
* `saturacion_centro = mean(centro > 0.95)`: mide si la región central ya tiene muchos píxeles cercanos al máximo antes del realce.

Estos indicadores modifican los parámetros del preprocesamiento:

* Si el centro ya está muy saturado, o si el fondo es muy intenso, se reduce el realce (`gain_log` menor, `clip_limit` bajo y kernel más grande).
* Si la masa se diferencia bien del fondo y hay poca competencia periférica, se permite un realce más fuerte (`gain_log` mayor, `clip_limit` mayor y kernel más chico).

Primero se aplica una transformación logarítmica, controlada por `gain_log`, para modificar la distribución de intensidades. Después se aplica CLAHE, controlado por `clip_limit` y `kernel_size`, para mejorar el contraste local. Finalmente, `preprocesar_miniroi_log_clahe` evalúa si el resultado quedó excesivamente saturado en la región central. Si eso ocurre, se activa el control anti-saturación y se recalcula el preprocesamiento con parámetros más suaves.


In [ ]:
# ============================================================
# CELDA 5 - Preprocesamiento local adaptativo miniroi_log_clahe
# Con control anti-saturación excepcional
# ============================================================

def mini_rois(img):
    h, w = img.shape

    # Mini-ROI central: zona probable de masa.
    cy, cx = h // 2, w // 2
    hc, wc = max(8, int(0.30 * h)), max(8, int(0.30 * w))

    centro = np.zeros_like(img, dtype=bool)
    centro[
        max(0, cy - hc // 2):min(h, cy + hc // 2),
        max(0, cx - wc // 2):min(w, cx + wc // 2)
    ] = True

    # Mini-ROIs de esquinas: aproximación de fondo local.
    he, we = max(6, int(0.18 * h)), max(6, int(0.18 * w))
    margen_y, margen_x = int(0.08 * h), int(0.08 * w)

    esquinas = []

    coords = [
        (margen_y, margen_x),
        (margen_y, w - margen_x - we),
        (h - margen_y - he, margen_x),
        (h - margen_y - he, w - margen_x - we)
    ]

    for y0, x0 in coords:
        m = np.zeros_like(img, dtype=bool)

        y0 = max(0, y0)
        x0 = max(0, x0)
        y1 = min(h, y0 + he)
        x1 = min(w, x0 + we)

        m[y0:y1, x0:x1] = True
        esquinas.append(m)

    return centro, esquinas


def elegir_parametros_miniroi(ventana):
    img = np.clip(ventana.astype(np.float32), 0, 1)

    centro, esquinas = mini_rois(img)

    vals_centro = img[centro]
    vals_fondo = np.concatenate([img[m] for m in esquinas])

    c50 = np.percentile(vals_centro, 50)
    c95 = np.percentile(vals_centro, 95)

    f25 = np.percentile(vals_fondo, 25)
    f50 = np.percentile(vals_fondo, 50)
    f75 = np.percentile(vals_fondo, 75)
    f90 = np.percentile(vals_fondo, 90)

    # Indicadores adaptativos.
    sep = (c50 - f50) / (f75 - f25 + 1e-8)
    comp = np.mean(vals_fondo > c50)
    sat = np.mean(vals_centro > 0.95)

    # Gain del log.
    if sat > 0.70:
        gain_log = 0.85
    elif f90 > 0.82 or comp > 0.20:
        gain_log = 0.95
    elif sep > 2.0 and comp < 0.10:
        gain_log = 1.20
    elif sep > 1.2 and comp < 0.15:
        gain_log = 1.10
    else:
        gain_log = 1.00

    gain_log = float(np.clip(gain_log, 0.80, 1.25))

    # CLAHE.
    if sat > 0.70 or f90 > 0.82 or comp > 0.20:
        clip_limit = 0.003
        kernel_div = 2
    elif sep > 2.0 and comp < 0.10:
        clip_limit = 0.008
        kernel_div = 4
    else:
        clip_limit = 0.005
        kernel_div = 3

    kernel_size = max(8, min(img.shape) // kernel_div)

    return {
        "gain_log": gain_log,
        "clip_limit": clip_limit,
        "kernel_size": kernel_size,
        "centro_p50": float(c50),
        "centro_p95": float(c95),
        "fondo_p50": float(f50),
        "fondo_p90": float(f90),
        "separacion": float(sep),
        "competencia_fondo": float(comp),
        "saturacion_centro": float(sat)
    }


def preprocesar_miniroi_log_clahe(ventana):
    # La ventana ya viene normalizada desde preparar_ventana_y_gt().
    # No se vuelve a normalizar por percentiles.
    ventana = np.clip(ventana.astype(np.float32), 0, 1)

    params = elegir_parametros_miniroi(ventana)

    # Primer preprocesamiento con parámetros adaptativos.
    img_log = exposure.adjust_log(
        ventana,
        gain=params["gain_log"]
    ).astype(np.float32)

    img_log = np.clip(img_log, 0, 1)

    img_clahe = exposure.equalize_adapthist(
        img_log,
        kernel_size=params["kernel_size"],
        clip_limit=params["clip_limit"]
    ).astype(np.float32)

    img_clahe = np.clip(img_clahe, 0, 1)

    # ------------------------------------------------------------
    # Control anti-saturación
    # ------------------------------------------------------------
    # Se activa sólo cuando el realce fue fuerte y el centro quedó
    # realmente quemado. Si se activa, se recalcula una versión más
    # suave; no se mezclan ambas imágenes.
    centro, _ = mini_rois(img_clahe)

    frac_centro_quemado = np.mean(img_clahe[centro] > 0.97)
    p95_centro_post = np.percentile(img_clahe[centro], 95)
    p99_centro_post = np.percentile(img_clahe[centro], 99)
    media_centro_post = np.mean(img_clahe[centro])

    preproc_fuerte = (
        params["gain_log"] >= 1.12 or
        params["clip_limit"] >= 0.007
    )

    activar_anti_saturacion = (
        preproc_fuerte and (
            (frac_centro_quemado > 0.68 and p99_centro_post > 0.998) or
            (frac_centro_quemado > 0.78) or
            (media_centro_post > 0.965 and p99_centro_post > 0.998)
        )
    )

    params["frac_centro_quemado_post"] = float(frac_centro_quemado)
    params["p95_centro_post"] = float(p95_centro_post)
    params["p99_centro_post"] = float(p99_centro_post)
    params["media_centro_post"] = float(media_centro_post)
    params["preproc_fuerte"] = bool(preproc_fuerte)
    params["anti_saturacion"] = bool(activar_anti_saturacion)

    if activar_anti_saturacion:
        gain_suave = min(params["gain_log"], 1.00)
        clip_suave = min(params["clip_limit"], 0.004)
        kernel_suave = max(params["kernel_size"], min(ventana.shape) // 2)

        img_log_suave = exposure.adjust_log(
            ventana,
            gain=gain_suave
        ).astype(np.float32)

        img_log_suave = np.clip(img_log_suave, 0, 1)

        img_final = exposure.equalize_adapthist(
            img_log_suave,
            kernel_size=kernel_suave,
            clip_limit=clip_suave
        ).astype(np.float32)

        img_final = np.clip(img_final, 0, 1)

        params["gain_log_usado"] = float(gain_suave)
        params["clip_limit_usado"] = float(clip_suave)
        params["kernel_size_usado"] = int(kernel_suave)

    else:
        img_final = img_clahe

        params["gain_log_usado"] = float(params["gain_log"])
        params["clip_limit_usado"] = float(params["clip_limit"])
        params["kernel_size_usado"] = int(params["kernel_size"])

    return img_final.astype(np.float32), params


### Visualización de mini-ROIs

Para seleccionar los parámetros del preprocesamiento se comparan intensidades de una mini-ROI central con mini-ROIs periféricas. La mini-ROI central aproxima la región donde se espera encontrar la masa, mientras que las periféricas se usan como referencia del fondo local.

A partir de esa comparación se calculan indicadores que modifican el `gain` de la transformación logarítmica, el `clip_limit` de CLAHE y el tamaño del kernel utilizado.

In [ ]:
# @title
# ============================================================
# Visualización - Mini-ROIs e indicadores numéricos
# ============================================================

from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

# Casos a visualizar
INDICES_MINIROI = [0, 1, 2, 3, 4, 5]
INDICES_MINIROI = [i for i in INDICES_MINIROI if i < len(df_casos_batch)]

# Colores suaves
COLOR_CENTRO = "#E9C46A"     # amarillo suave
COLOR_FONDO = "#8FBBD9"      # celeste suave
COLOR_GT = "#B07A8C"         # malva suave


def bbox_desde_mask(mask):
    y, x = np.where(mask)

    if len(y) == 0:
        return None

    return (
        int(x.min()),
        int(y.min()),
        int(x.max() - x.min() + 1),
        int(y.max() - y.min() + 1)
    )


# ============================================================
# Leyenda única
# ============================================================

handles = [
    Rectangle(
        (0, 0), 1, 1,
        fill=False,
        edgecolor=COLOR_CENTRO,
        linewidth=2,
        label="Mini-ROI central"
    ),
    Rectangle(
        (0, 0), 1, 1,
        fill=False,
        edgecolor=COLOR_FONDO,
        linewidth=2,
        label="Mini-ROIs periféricas"
    ),
    Line2D(
        [0], [0],
        color=COLOR_GT,
        linewidth=2,
        label="Máscara de referencia"
    )
]

fig_leg = plt.figure(figsize=(8, 0.8))
fig_leg.legend(
    handles=handles,
    loc="center",
    ncol=3,
    frameon=True,
    fontsize=9
)
plt.axis("off")

plt.savefig(
    FIG_DIR / "leyenda_minirois.png",
    dpi=180,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# Figuras e indicadores
# ============================================================

filas_indicadores = []

for idx in INDICES_MINIROI:

    caso = df_casos_batch.iloc[idx]

    ventana, gt, info = preparar_ventana_y_gt(caso)
    ventana_pre, params = preprocesar_miniroi_log_clahe(ventana)

    centro, esquinas = mini_rois(ventana)

    fig, axes = plt.subplots(
        1, 2,
        figsize=(11, 5),
        constrained_layout=True
    )

    fig.suptitle(
        f"Mini-ROIs para ajuste adaptativo — {caso['patient_id']} | {caso['lateralidad']} | {caso['vista']}",
        fontsize=13
    )

    # ------------------------------------------------------------
    # 1. Ventana normalizada + mini-ROIs
    # ------------------------------------------------------------
    axes[0].imshow(ventana, cmap="gray", vmin=0, vmax=1)

    bbox_centro = bbox_desde_mask(centro)

    if bbox_centro is not None:
        x0, y0, ancho, alto = bbox_centro
        axes[0].add_patch(
            Rectangle(
                (x0, y0),
                ancho,
                alto,
                fill=False,
                edgecolor=COLOR_CENTRO,
                linewidth=2.2
            )
        )

    for m in esquinas:
        bbox = bbox_desde_mask(m)

        if bbox is not None:
            x0, y0, ancho, alto = bbox
            axes[0].add_patch(
                Rectangle(
                    (x0, y0),
                    ancho,
                    alto,
                    fill=False,
                    edgecolor=COLOR_FONDO,
                    linewidth=2.2
                )
            )

    axes[0].contour(
        gt,
        levels=[0.5],
        colors=[COLOR_GT],
        linewidths=1.8
    )

    axes[0].set_title("1. Ventana normalizada + mini-ROIs", fontsize=11, pad=10)
    axes[0].axis("off")

    # ------------------------------------------------------------
    # 2. Ventana preprocesada
    # ------------------------------------------------------------
    axes[1].imshow(ventana_pre, cmap="gray", vmin=0, vmax=1)
    axes[1].contour(
        gt,
        levels=[0.5],
        colors=[COLOR_GT],
        linewidths=1.8
    )

    axes[1].set_title(
        "2. Ventana preprocesada\n"
        f"gain_log={params['gain_log_usado']:.2f} | "
        f"clip={params['clip_limit_usado']:.3f} | "
        f"kernel={params['kernel_size_usado']}",
        fontsize=11,
        pad=10
    )
    axes[1].axis("off")

    plt.savefig(
        FIG_DIR / f"mini_rois_{idx:03d}_{caso['case_id']}.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    filas_indicadores.append({
        "idx": idx,
        "case_id": caso["case_id"],
        "patient_id": caso["patient_id"],
        "vista": caso["vista"],
        "mass_shape": caso["mass_shape"],
        "mass_margins": caso["mass_margins"],
        "breast_density": caso["breast_density"],
        "centro_p50": params["centro_p50"],
        "centro_p95": params["centro_p95"],
        "fondo_p50": params["fondo_p50"],
        "fondo_p90": params["fondo_p90"],
        "separacion": params["separacion"],
        "competencia_fondo": params["competencia_fondo"],
        "saturacion_centro": params["saturacion_centro"],
        "gain_log": params["gain_log"],
        "clip_limit": params["clip_limit"],
        "kernel_size": params["kernel_size"],
        "anti_saturacion": params["anti_saturacion"],
        "gain_log_usado": params["gain_log_usado"],
        "clip_limit_usado": params["clip_limit_usado"],
        "kernel_size_usado": params["kernel_size_usado"]
    })


df_indicadores_miniroi = pd.DataFrame(filas_indicadores)

df_indicadores_miniroi.to_csv(
    OUT_DIR / "indicadores_miniroi_casos_representativos.csv",
    index=False
)

print("Indicadores y parámetros adaptativos de los casos visualizados:")
display(
    df_indicadores_miniroi[[
        "idx",
        "case_id",
        "vista",
        "centro_p50",
        "fondo_p50",
        "fondo_p90",
        "separacion",
        "competencia_fondo",
        "saturacion_centro",
        "gain_log",
        "clip_limit",
        "kernel_size",
        "anti_saturacion",
        "gain_log_usado",
        "clip_limit_usado",
        "kernel_size_usado"
    ]]
    .round(4)
    .reset_index(drop=True)
)

print("\nTabla completa guardada en:")
print(OUT_DIR / "indicadores_miniroi_casos_representativos.csv")

### Visualización de las etapas del preprocesamiento

Las figuras siguientes muestran tanto las imágenes como sus histogramas, para visualizar cómo cada etapa modifica la distribución de intensidades.


In [ ]:
# @title
# ============================================================
# Helper - Reconstrucción de etapas del preprocesamiento
# ============================================================

from skimage import exposure

def obtener_ventana_raw_norm_gt(caso):
    """
    Devuelve:
    - ventana_raw: recorte original desde la mamografía
    - ventana_norm: ventana normalizada entre 0 y 1
    - gt_ventana: máscara de referencia en la ventana
    """

    mg = leer_mamografia(caso["ruta_mg"])
    roi_raw = leer_dicom_imagen(caso["ruta_roi"])
    mask_raw = leer_dicom_imagen(caso["ruta_mask"])

    roi_2d = np.squeeze(roi_raw)
    mask_full = redimensionar_mask_a_mg(mask_raw, mg.shape).astype(bool)

    y, x = np.where(mask_full)

    if len(y) == 0:
        raise ValueError("La máscara de referencia está vacía.")

    cy = int((y.min() + y.max()) / 2)
    cx = int((x.min() + x.max()) / 2)

    alto_roi, ancho_roi = roi_2d.shape

    wy0, wy1, wx0, wx1 = calcular_ventana_centrada(
        cy,
        cx,
        alto_roi,
        ancho_roi,
        mg.shape
    )

    ventana_raw = mg[wy0:wy1, wx0:wx1]
    ventana_norm = normalizar_ventana_local(ventana_raw)
    ventana_norm = np.clip(ventana_norm, 0, 1)

    gt_ventana = mask_full[wy0:wy1, wx0:wx1]

    return ventana_raw, ventana_norm, gt_ventana


def reconstruir_etapas_preprocesamiento(caso):
    """
    Reconstruye las etapas del preprocesamiento para visualización.

    Importante:
    Se usa np.clip después de cada operación para asegurar que las imágenes
    float queden en rango [0, 1], que es el rango esperado por equalize_adapthist.
    """

    ventana_raw, ventana_norm, gt = obtener_ventana_raw_norm_gt(caso)

    # Resultado final del pipeline y parámetros elegidos
    ventana_final, params = preprocesar_miniroi_log_clahe(ventana_norm)
    ventana_final = np.clip(ventana_final, 0, 1)

    # Etapa intermedia: transformación logarítmica con parámetros iniciales
    ventana_log = exposure.adjust_log(
        ventana_norm,
        gain=params["gain_log"]
    )
    ventana_log = np.clip(ventana_log, 0, 1)

    # Etapa intermedia: log + CLAHE con parámetros iniciales
    ventana_log_clahe_inicial = exposure.equalize_adapthist(
        ventana_log,
        clip_limit=params["clip_limit"],
        kernel_size=params["kernel_size"]
    )
    ventana_log_clahe_inicial = np.clip(ventana_log_clahe_inicial, 0, 1)

    return {
        "ventana_raw": ventana_raw,
        "ventana_norm": ventana_norm,
        "ventana_log": ventana_log,
        "ventana_log_clahe_inicial": ventana_log_clahe_inicial,
        "ventana_final": ventana_final,
        "gt": gt,
        "params": params
    }

In [ ]:
# @title
# ============================================================
# Visualización - Etapas del preprocesamiento con histogramas
# ============================================================

IDX_VIS_PRE = [0, 1, 16, 2, 20, 5]
IDX_VIS_PRE = [i for i in IDX_VIS_PRE if i < len(df_casos_batch)]

COLOR_GT = "#B07A8C"

COL_HIST_RAW   = "#B0B0B0"   # gris suave
COL_HIST_NORM  = "#8FBBD9"   # celeste suave
COL_HIST_LOG   = "#A8C686"   # verde suave
COL_HIST_CLAHE = "#CDB4DB"   # lila suave
COL_HIST_FINAL = "#D8A7A7"   # salmón suave

filas_params_pre = []

for idx in IDX_VIS_PRE:

    caso = df_casos_batch.iloc[idx]
    etapas = reconstruir_etapas_preprocesamiento(caso)

    ventana_raw = etapas["ventana_raw"]
    ventana_norm = etapas["ventana_norm"]
    ventana_log = etapas["ventana_log"]
    ventana_log_clahe_inicial = etapas["ventana_log_clahe_inicial"]
    ventana_final = etapas["ventana_final"]
    gt = etapas["gt"]
    params = etapas["params"]

    vals_raw = ventana_raw[np.isfinite(ventana_raw)]
    vmin_raw, vmax_raw = np.percentile(vals_raw, [1, 99])

    fig, axes = plt.subplots(
        2,
        5,
        figsize=(20, 7.5),
        constrained_layout=False
    )

    fig.suptitle(
        f"Etapas del preprocesamiento — {caso['patient_id']} | {caso['lateralidad']} | {caso['vista']}",
        fontsize=14,
        y=0.98
    )

    imagenes = [
        (ventana_raw, "1. Ventana cruda", "raw"),
        (ventana_norm, "2. Normalizada", "norm"),
        (ventana_log, "3. Log", "log"),
        (ventana_log_clahe_inicial, "4. Log + CLAHE inicial", "clahe"),
        (ventana_final, "5. Resultado final", "final"),
    ]

    colores_hist = {
        "raw": COL_HIST_RAW,
        "norm": COL_HIST_NORM,
        "log": COL_HIST_LOG,
        "clahe": COL_HIST_CLAHE,
        "final": COL_HIST_FINAL
    }

    for j, (img, titulo, clave) in enumerate(imagenes):

        ax_img = axes[0, j]

        if clave == "raw":
            ax_img.imshow(img, cmap="gray", vmin=vmin_raw, vmax=vmax_raw)
        else:
            ax_img.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax_img.contour(gt, levels=[0.5], colors=[COLOR_GT], linewidths=1.6)

        ax_img.set_title(titulo, fontsize=11, pad=8)
        ax_img.axis("off")

        ax_hist = axes[1, j]
        datos = img[np.isfinite(img)].ravel()

        if clave == "raw":
            ax_hist.hist(datos, bins=80, color=colores_hist[clave], alpha=0.9)
            ax_hist.set_xlim(np.percentile(datos, 0.5), np.percentile(datos, 99.5))
        else:
            datos = np.clip(datos, 0, 1)
            ax_hist.hist(datos, bins=80, range=(0, 1), color=colores_hist[clave], alpha=0.9)
            ax_hist.set_xlim(0, 1)

        ax_hist.set_title("Histograma", fontsize=10)
        ax_hist.set_xlabel("Intensidad", fontsize=9)
        ax_hist.set_ylabel("Frecuencia", fontsize=9)
        ax_hist.tick_params(labelsize=8)

    fig.subplots_adjust(
        top=0.88,
        bottom=0.10,
        left=0.04,
        right=0.98,
        hspace=0.28,
        wspace=0.22
    )

    plt.savefig(
        FIG_DIR / f"preprocesamiento_etapas_hist_{idx:03d}_{caso['case_id']}.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    filas_params_pre.append({
        "idx": idx,
        "case_id": caso["case_id"],
        "patient_id": caso["patient_id"],
        "lateralidad": caso["lateralidad"],
        "vista": caso["vista"],
        "separacion": params["separacion"],
        "competencia_fondo": params["competencia_fondo"],
        "saturacion_centro": params["saturacion_centro"],
        "gain_log": params["gain_log"],
        "gain_log_usado": params["gain_log_usado"],
        "clip_limit": params["clip_limit"],
        "clip_limit_usado": params["clip_limit_usado"],
        "kernel_size": params["kernel_size"],
        "kernel_size_usado": params["kernel_size_usado"],
        "frac_centro_quemado_post": params["frac_centro_quemado_post"],
        "p99_centro_post": params["p99_centro_post"],
        "anti_saturacion": params["anti_saturacion"]
    })


df_params_preprocesamiento = pd.DataFrame(filas_params_pre)

print("Indicadores y parámetros usados en los casos visualizados:")
display(
    df_params_preprocesamiento[[
        "idx",
        "case_id",
        "vista",
        "separacion",
        "competencia_fondo",
        "saturacion_centro",
        "gain_log",
        "gain_log_usado",
        "clip_limit",
        "clip_limit_usado",
        "kernel_size",
        "kernel_size_usado",
        "anti_saturacion"
    ]]
    .round(4)
    .reset_index(drop=True)
)

df_params_preprocesamiento.to_csv(
    OUT_DIR / "parametros_preprocesamiento_casos_visualizados.csv",
    index=False
)

### Revisión del control anti-saturación

El control anti-saturación no se aplica siempre. Primero se realiza una versión inicial del preprocesamiento y luego se evalúa si la región central quedó demasiado saturada.

Para eso se revisan dos indicadores: `frac_centro_quemado_post`, que mide la proporción de píxeles centrales con intensidades muy altas después del realce, y `p99_centro_post`, que representa el percentil 99 de la región central. Si estos valores superan el criterio definido en la función, se reduce el realce usando valores menores de `gain_log`, `clip_limit` y tamaño del kernel.

Esta celda resume en cuántos casos se activó ese ajuste y guarda la tabla completa de diagnóstico en Drive.

In [ ]:
# @title
# ============================================================
# CELDA EXTRA - Frecuencia de activación del control anti-saturación
# ============================================================

filas_anti = []

# Esta revisión aplica sólo el preprocesamiento; no vuelve a segmentar.
df_revision = df_casos_ok.copy()

for i, (_, caso) in enumerate(df_revision.iterrows()):
    try:
        ventana, gt, info = preparar_ventana_y_gt(caso)
        ventana_pre, params = preprocesar_miniroi_log_clahe(ventana)

        filas_anti.append({
            "idx": i,
            "case_id": caso["case_id"],
            "patient_id": caso["patient_id"],
            "lateralidad": caso["lateralidad"],
            "vista": caso["vista"],
            "mass_shape": caso["mass_shape"],
            "mass_margins": caso["mass_margins"],
            "breast_density": caso["breast_density"],
            "anti_saturacion": params["anti_saturacion"],
            "frac_centro_quemado_post": params["frac_centro_quemado_post"],
            "p99_centro_post": params["p99_centro_post"],
            "gain_log": params["gain_log"],
            "gain_log_usado": params["gain_log_usado"],
            "clip_limit": params["clip_limit"],
            "clip_limit_usado": params["clip_limit_usado"],
            "kernel_size": params["kernel_size"],
            "kernel_size_usado": params["kernel_size_usado"],
            "error": None
        })

    except Exception as e:
        filas_anti.append({
            "idx": i,
            "case_id": caso.get("case_id", np.nan),
            "patient_id": caso.get("patient_id", np.nan),
            "lateralidad": caso.get("lateralidad", np.nan),
            "vista": caso.get("vista", np.nan),
            "mass_shape": caso.get("mass_shape", np.nan),
            "mass_margins": caso.get("mass_margins", np.nan),
            "breast_density": caso.get("breast_density", np.nan),
            "anti_saturacion": np.nan,
            "frac_centro_quemado_post": np.nan,
            "p99_centro_post": np.nan,
            "gain_log": np.nan,
            "gain_log_usado": np.nan,
            "clip_limit": np.nan,
            "clip_limit_usado": np.nan,
            "kernel_size": np.nan,
            "kernel_size_usado": np.nan,
            "error": str(e)
        })

    finally:
        for nombre_var in ["ventana", "ventana_pre", "gt", "info", "params"]:
            if nombre_var in locals():
                del locals()[nombre_var]

        plt.close("all")
        gc.collect()


df_anti = pd.DataFrame(filas_anti)
df_anti.to_csv(OUT_DIR / "diagnostico_anti_saturacion.csv", index=False)

n_total = len(df_anti)
n_errores = int(df_anti["error"].notna().sum())
n_validos = n_total - n_errores
n_anti = int(df_anti["anti_saturacion"].fillna(False).sum())

print("Cantidad total de casos/vistas revisados:", n_total)
print("Casos procesados correctamente:", n_validos)
print("Casos con error:", n_errores)
print("Cantidad con anti-saturación activada:", n_anti)

if n_validos > 0:
    print("Porcentaje con anti-saturación activada:", round(100 * n_anti / n_validos, 2), "%")

if n_anti > 0:
    print("\nCasos donde se activó anti-saturación:")
    display(
        df_anti[df_anti["anti_saturacion"] == True][[
            "idx",
            "case_id",
            "patient_id",
            "lateralidad",
            "vista",
            "mass_shape",
            "mass_margins",
            "breast_density",
            "frac_centro_quemado_post",
            "p99_centro_post",
            "gain_log",
            "gain_log_usado",
            "clip_limit",
            "clip_limit_usado"
        ]]
        .round(4)
        .reset_index(drop=True)
    )
else:
    print("\nNo se activó anti-saturación en los casos revisados.")

if n_errores > 0:
    print("\nCasos con error durante la revisión:")
    display(
        df_anti[df_anti["error"].notna()][[
            "idx",
            "case_id",
            "patient_id",
            "lateralidad",
            "vista",
            "error"
        ]]
        .reset_index(drop=True)
    )

print("\nTabla completa guardada en:")
print(OUT_DIR / "diagnostico_anti_saturacion.csv")

In [ ]:
# @title
# ============================================================
# Visualización - Casos con anti-saturación activada
# ============================================================

if "df_anti" not in globals():
    raise ValueError("Primero corré la celda de diagnóstico de anti-saturación para generar df_anti.")

idx_anti = df_anti.loc[df_anti["anti_saturacion"] == True, "idx"].tolist()

COLOR_GT = "#B07A8C"

COL_HIST_NORM  = "#8FBBD9"
COL_HIST_LOG   = "#A8C686"
COL_HIST_CLAHE = "#CDB4DB"
COL_HIST_FINAL = "#D8A7A7"

filas_params_anti = []

if len(idx_anti) == 0:
    print("No hay casos con anti-saturación activada para visualizar.")

else:
    print("Casos con anti-saturación activada:", idx_anti)

    for idx in idx_anti[:6]:

        caso = df_casos_ok.iloc[idx]
        etapas = reconstruir_etapas_preprocesamiento(caso)

        ventana_norm = np.clip(etapas["ventana_norm"], 0, 1)
        ventana_log = np.clip(etapas["ventana_log"], 0, 1)
        ventana_log_clahe_inicial = np.clip(etapas["ventana_log_clahe_inicial"], 0, 1)
        ventana_final = np.clip(etapas["ventana_final"], 0, 1)
        gt = etapas["gt"]
        params = etapas["params"]

        fig, axes = plt.subplots(
            2,
            4,
            figsize=(17, 7.5),
            constrained_layout=False
        )

        fig.suptitle(
            f"Control anti-saturación — {caso['patient_id']} | {caso['lateralidad']} | {caso['vista']}",
            fontsize=14,
            y=0.98
        )

        imagenes = [
            (ventana_norm, "1. Normalizada", COL_HIST_NORM),
            (ventana_log, "2. Log", COL_HIST_LOG),
            (ventana_log_clahe_inicial, "3. Antes del ajuste", COL_HIST_CLAHE),
            (ventana_final, "4. Después del ajuste", COL_HIST_FINAL),
        ]

        for j, (img, titulo, color_hist) in enumerate(imagenes):

            ax_img = axes[0, j]
            ax_img.imshow(img, cmap="gray", vmin=0, vmax=1)
            ax_img.contour(gt, levels=[0.5], colors=[COLOR_GT], linewidths=1.6)
            ax_img.set_title(titulo, fontsize=11, pad=8)
            ax_img.axis("off")

            ax_hist = axes[1, j]
            datos = np.clip(img[np.isfinite(img)].ravel(), 0, 1)

            ax_hist.hist(datos, bins=80, range=(0, 1), color=color_hist, alpha=0.9)
            ax_hist.set_xlim(0, 1)
            ax_hist.set_title("Histograma", fontsize=10)
            ax_hist.set_xlabel("Intensidad", fontsize=9)
            ax_hist.set_ylabel("Frecuencia", fontsize=9)
            ax_hist.tick_params(labelsize=8)

        fig.subplots_adjust(
            top=0.88,
            bottom=0.10,
            left=0.05,
            right=0.98,
            hspace=0.28,
            wspace=0.22
        )

        plt.savefig(
            FIG_DIR / f"antisaturacion_{idx:03d}_{caso['case_id']}.png",
            dpi=180,
            bbox_inches="tight"
        )

        plt.show()

        filas_params_anti.append({
            "idx": idx,
            "case_id": caso["case_id"],
            "patient_id": caso["patient_id"],
            "lateralidad": caso["lateralidad"],
            "vista": caso["vista"],
            "frac_centro_quemado_post": params["frac_centro_quemado_post"],
            "p99_centro_post": params["p99_centro_post"],
            "gain_log": params["gain_log"],
            "gain_log_usado": params["gain_log_usado"],
            "clip_limit": params["clip_limit"],
            "clip_limit_usado": params["clip_limit_usado"],
            "kernel_size": params["kernel_size"],
            "kernel_size_usado": params["kernel_size_usado"]
        })


    df_params_anti = pd.DataFrame(filas_params_anti)

    print("Parámetros de los casos visualizados con anti-saturación:")
    display(
        df_params_anti.round(4).reset_index(drop=True)
    )

    df_params_anti.to_csv(
        OUT_DIR / "parametros_antisaturacion_casos_visualizados.csv",
        index=False
    )

# *3. Segmentación*



En esta celda se definen las funciones principales de segmentación que se aplican sobre la ventana ya preprocesada. Todas reciben una imagen 2D normalizada y devuelven máscaras binarias, donde los píxeles con valor verdadero corresponden a la región segmentada como masa.

La función `segmentar_gac` implementa el método de contorno activo geodésico morfológico. Primero suaviza la imagen con un filtro gaussiano y luego calcula el `inverse_gaussian_gradient`, que genera una imagen donde los bordes funcionan como zonas de detención del contorno. Se usan los parámetros `alpha` y `sigma`, que regulan la sensibilidad y el suavizado del mapa de bordes. El contorno se inicializa con una máscara circular centrada en la ventana. Luego evoluciona durante `num_iter` iteraciones, `smoothing` suaviza el contorno, y `balloon` favorece su expansión.

Después de obtener la máscara de GAC, se aplican funciones auxiliares: `limpiar_mask` elimina regiones pequeñas, rellena huecos y conserva la componente principal; `corregir_gac_por_puente_filtrado` busca unir regiones de la máscara con huecos que se generaron por mínimos locales; `recortar_local_si_toca_borde` se usa cuando la máscara toca el borde de la ventana, recortandola.

La función `segmentar_kmeans` aplica K-means sobre la ventana preprocesada. En este caso, cada píxel se representa usando su intensidad y su cercanía al centro de la ventana. Se usa `n_clusters=3`, `random_state=0` y `n_init=10` para encontrar la mejor inicialización de los centrides para los clusters. Además, se utiliza un parámetro de peso para la distancia al centro, de modo que el cluster elegido no dependa sólo de la intensidad sino también de la ubicación esperada de la masa.

La función `segmentar_hibrido` combina las máscaras obtenidas con GAC y K-means. Toma la máscara de GAC como base, genera una banda cercana mediante dilatación morfológica y agrega sólo los píxeles de K-means que caen dentro de esa región. El parámetro principal de esta parte es el radio de la banda (`RADIO_BANDA_HIBRIDO`), que define qué tan lejos del contorno de GAC se permite incorporar píxeles de la máscara de K-means.

Por último, `segmentar_pipeline` reúne todo el proceso: aplica GAC, aplica K-means y genera la máscara híbrida. La salida de esta función son las tres máscaras (`seg_gac`, `seg_kmeans` y `seg_hibrida`), que después se comparan con la máscara de referencia mediante métricas de segmentación.


In [ ]:
# ============================================================
# CELDA 6 - Segmentación GAC + K-means + híbrido
# Con puente filtrado + recorte local condicional en bordes
# ============================================================

RADIO_BANDA_HIBRIDO = 6

# Activar/desactivar puente sobre GAC.
USAR_PUENTE_GAC = True

# Parámetros del puente GAC.
RADIO_BOCA_GAC = 35
AREA_MIN_PUENTE_GAC = 500
AREA_MAX_PUENTE_GAC = 8000

# Activar/desactivar control de borde sobre GAC.
USAR_CONTROL_BORDE_GAC = True

# Parámetros del control de borde.
MARGEN_BORDE_GAC = 3
ANCHO_RECORTE_BORDE_GAC = 28
MIN_AREA_REL_POST_RECORTE = 0.45


def limpiar_mask(mask, min_size=50):
    mask = mask.astype(bool)
    mask = morphology.remove_small_objects(mask, min_size=min_size)
    mask = ndi.binary_fill_holes(mask)
    return mask.astype(bool)


def toca_borde(mask, margen=3):
    """
    Devuelve True si la máscara toca alguno de los bordes de la ventana.
    """
    mask = mask.astype(bool)

    if mask.sum() == 0:
        return False

    return (
        mask[:margen, :].any() or
        mask[-margen:, :].any() or
        mask[:, :margen].any() or
        mask[:, -margen:].any()
    )


def lados_que_toca_borde(mask, margen=3):
    mask = mask.astype(bool)

    return {
        "arriba": bool(mask[:margen, :].any()),
        "abajo": bool(mask[-margen:, :].any()),
        "izquierda": bool(mask[:, :margen].any()),
        "derecha": bool(mask[:, -margen:].any())
    }


def recortar_local_si_toca_borde(mask, margen=3, ancho_recorte=18, min_area_rel=0.45):
    """
    Si la máscara toca algún borde de la ventana, recorta solamente una franja
    del lado donde toca. No erosiona toda la masa.

    Se prueban distintos anchos de recorte y se acepta el primero que:
    - deja de tocar el borde;
    - conserva una fracción mínima del área original.
    """

    mask = mask.astype(bool)
    lados = lados_que_toca_borde(mask, margen=margen)

    if not any(lados.values()):
        return mask.astype(bool), False, lados

    area_original = mask.sum()

    if area_original == 0:
        return mask.astype(bool), False, lados

    anchos_a_probar = [
        ancho_recorte,
        max(margen + 2, ancho_recorte // 2),
        max(margen + 1, ancho_recorte // 3)
    ]

    mejor_candidato = mask.copy()
    mejor_area = 0

    for ancho in anchos_a_probar:
        zona_recorte = np.zeros_like(mask, dtype=bool)

        if lados["arriba"]:
            zona_recorte[:ancho, :] = True
        if lados["abajo"]:
            zona_recorte[-ancho:, :] = True
        if lados["izquierda"]:
            zona_recorte[:, :ancho] = True
        if lados["derecha"]:
            zona_recorte[:, -ancho:] = True

        candidato = mask & (~zona_recorte)
        candidato = morphology.remove_small_objects(candidato, min_size=50)
        candidato = ndi.binary_fill_holes(candidato)

        area_candidato = candidato.sum()
        deja_de_tocar = not toca_borde(candidato, margen=margen)

        if area_candidato > mejor_area and deja_de_tocar:
            mejor_candidato = candidato.copy()
            mejor_area = area_candidato

        if area_candidato >= min_area_rel * area_original and deja_de_tocar:
            return candidato.astype(bool), True, lados

    if mejor_area >= min_area_rel * area_original:
        return mejor_candidato.astype(bool), True, lados

    # Si todos los recortes destruyen demasiado la máscara, no se corrige.
    return mask.astype(bool), False, lados


def corregir_gac_por_puente_filtrado(
    seg,
    radio_boca=40,
    area_min=500,
    area_max=8000,
    devolver_info=False
):
    """
    Cierra entradas o bocas negras del GAC y rellena huecos, pero conserva
    sólo los píxeles agregados cuyo tamaño se encuentra dentro de un rango.
    """

    seg = seg.astype(bool)

    info = {
        "puente_aplicado": False,
        "pixeles_puente_agregados": 0
    }

    if seg.sum() == 0:
        return (seg, info) if devolver_info else seg

    seg_puente = morphology.binary_closing(
        seg,
        morphology.disk(radio_boca)
    )

    seg_rellena = ndi.binary_fill_holes(seg_puente)
    agregados = seg_rellena & (~seg)

    labels_agregados = measure.label(agregados)
    props = measure.regionprops(labels_agregados)

    agregados_filtrados = np.zeros_like(seg, dtype=bool)

    for p in props:
        if area_min <= p.area <= area_max:
            agregados_filtrados |= labels_agregados == p.label

    seg_final = seg | agregados_filtrados
    seg_final = limpiar_mask(seg_final)

    info["pixeles_puente_agregados"] = int(agregados_filtrados.sum())
    info["puente_aplicado"] = bool(agregados_filtrados.any())

    return (seg_final, info) if devolver_info else seg_final


def segmentar_gac(img, devolver_info=False):
    h, w = img.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.ogrid[:h, :w]

    img_suave = filters.gaussian(img, sigma=0.8)
    gradiente = inverse_gaussian_gradient(img_suave, alpha=60, sigma=1.5)

    radio_ini = max(5, int(0.12 * min(h, w)))
    inicial = ((yy - cy)**2 + (xx - cx)**2) <= radio_ini**2

    seg = morphological_geodesic_active_contour(
        gradiente,
        num_iter=105,
        init_level_set=inicial,
        smoothing=1,
        balloon=1.0
    )

    seg = limpiar_mask(seg)

    info_gac = {
        "radio_inicial_gac": int(radio_ini),
        "area_gac_crudo": int(seg.sum()),
        "puente_aplicado": False,
        "pixeles_puente_agregados": 0,
        "borde_corregido": False,
        "toca_arriba": False,
        "toca_abajo": False,
        "toca_izquierda": False,
        "toca_derecha": False
    }

    # Corrección por puente filtrado.
    if USAR_PUENTE_GAC:
        seg, info_puente = corregir_gac_por_puente_filtrado(
            seg,
            radio_boca=RADIO_BOCA_GAC,
            area_min=AREA_MIN_PUENTE_GAC,
            area_max=AREA_MAX_PUENTE_GAC,
            devolver_info=True
        )
        info_gac.update(info_puente)

    # Recorte local condicional del lado que toca el borde.
    if USAR_CONTROL_BORDE_GAC:
        seg, borde_corregido, lados_borde = recortar_local_si_toca_borde(
            seg,
            margen=MARGEN_BORDE_GAC,
            ancho_recorte=ANCHO_RECORTE_BORDE_GAC,
            min_area_rel=MIN_AREA_REL_POST_RECORTE
        )

        info_gac["borde_corregido"] = bool(borde_corregido)
        info_gac["toca_arriba"] = lados_borde["arriba"]
        info_gac["toca_abajo"] = lados_borde["abajo"]
        info_gac["toca_izquierda"] = lados_borde["izquierda"]
        info_gac["toca_derecha"] = lados_borde["derecha"]

    info_gac["area_gac_final"] = int(seg.sum())

    if devolver_info:
        return seg.astype(bool), info_gac

    return seg.astype(bool)


def segmentar_kmeans(img):
    h, w = img.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.ogrid[:h, :w]

    img_suave = filters.gaussian(img, sigma=1.2)

    dist = np.sqrt((yy - cy)**2 + (xx - cx)**2)
    dist = dist / (dist.max() + 1e-8)

    peso_distancia = 0.10

    datos = np.column_stack([
        img_suave.ravel(),
        peso_distancia * (1 - dist.ravel())
    ])

    kmeans = KMeans(n_clusters=3, random_state=0, n_init=10)
    labels = kmeans.fit_predict(datos).reshape(h, w)

    scores = []

    for lab in range(3):
        region = labels == lab

        if region.sum() == 0:
            scores.append(-np.inf)
        else:
            intensidad = img_suave[region].mean()
            cercania = (1 - dist[region]).mean()
            scores.append(intensidad + peso_distancia * cercania)

    cluster_masa = int(np.argmax(scores))
    seg = labels == cluster_masa

    return limpiar_mask(seg)


def segmentar_hibrido(seg_gac, seg_kmeans, radio_banda=6):
    seg_gac = seg_gac.astype(bool)
    seg_kmeans = seg_kmeans.astype(bool)

    zona_permitida = morphology.binary_dilation(
        seg_gac,
        morphology.disk(radio_banda)
    )

    agregado = seg_kmeans & zona_permitida
    seg = seg_gac | agregado

    lab = measure.label(seg)
    labels_con_gac = np.unique(lab[seg_gac & (lab > 0)])

    if len(labels_con_gac) > 0:
        seg = np.isin(lab, labels_con_gac)
    else:
        seg = seg_gac.copy()

    return limpiar_mask(seg)


def segmentar_pipeline(img_pre, devolver_info=False):
    seg_gac, info_gac = segmentar_gac(img_pre, devolver_info=True)
    seg_kmeans = segmentar_kmeans(img_pre)

    seg_hibrida = segmentar_hibrido(
        seg_gac,
        seg_kmeans,
        radio_banda=RADIO_BANDA_HIBRIDO
    )

    if devolver_info:
        return seg_gac, seg_kmeans, seg_hibrida, info_gac

    return seg_gac, seg_kmeans, seg_hibrida


### Visualización de MorphGAC

Esta celda muestra las etapas principales del método MorphGAC. Primero se visualiza la ventana preprocesada y el gradiente gaussiano inverso, que es la imagen sobre la que evoluciona el contorno. Luego se muestra la inicialización circular, la máscara obtenida directamente por GAC y la máscara final después de las correcciones morfológicas.


In [ ]:
# @title
# ============================================================
# Visualización - Etapas principales de MorphGAC
# ============================================================

from matplotlib.patches import Circle

IDX_VIS_GAC = [4, 5, 8, 16, 27, 30]
IDX_VIS_GAC = [i for i in IDX_VIS_GAC if i < len(df_casos_batch)]

# Colores suaves
COLOR_INIT = "#E9C46A"      # amarillo suave
COLOR_MASK = "#7E9F90"      # verde grisáceo suave

filas_gac_visual = []

for idx in IDX_VIS_GAC:

    caso = df_casos_batch.iloc[idx]

    ventana, gt, info_ventana = preparar_ventana_y_gt(caso)
    ventana_pre, params_pre = preprocesar_miniroi_log_clahe(ventana)

    h, w = ventana_pre.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.ogrid[:h, :w]

    # ------------------------------------------------------------
    # Etapas internas de MorphGAC
    # ------------------------------------------------------------
    img_suave = filters.gaussian(ventana_pre, sigma=0.8)

    gradiente_inverso = inverse_gaussian_gradient(
        img_suave,
        alpha=60,
        sigma=1.5
    )

    radio_ini = max(5, int(0.12 * min(h, w)))

    inicial = (
        (yy - cy) ** 2 +
        (xx - cx) ** 2
    ) <= radio_ini ** 2

    gac_crudo = morphological_geodesic_active_contour(
        gradiente_inverso,
        num_iter=105,
        init_level_set=inicial,
        smoothing=1,
        balloon=1.0
    ).astype(bool)

    gac_final = limpiar_mask(gac_crudo)

    if USAR_PUENTE_GAC:
        gac_final = corregir_gac_por_puente_filtrado(
            gac_final,
            radio_boca=RADIO_BOCA_GAC,
            area_min=AREA_MIN_PUENTE_GAC,
            area_max=AREA_MAX_PUENTE_GAC
        )

    if USAR_CONTROL_BORDE_GAC:
        gac_final, borde_corregido, lados_borde = recortar_local_si_toca_borde(
            gac_final,
            margen=MARGEN_BORDE_GAC,
            ancho_recorte=ANCHO_RECORTE_BORDE_GAC,
            min_area_rel=MIN_AREA_REL_POST_RECORTE
        )
    else:
        borde_corregido = False
        lados_borde = {}

    # ------------------------------------------------------------
    # Figura
    # ------------------------------------------------------------
    fig, axes = plt.subplots(
        1,
        5,
        figsize=(20, 4.8),
        constrained_layout=False
    )

    fig.suptitle(
        f"Etapas de MorphGAC — {caso['patient_id']} | {caso['lateralidad']} | {caso['vista']}",
        fontsize=14,
        y=0.98
    )

    # 1. Ventana preprocesada
    axes[0].imshow(ventana_pre, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("1. Ventana preprocesada", fontsize=11, pad=8)
    axes[0].axis("off")

    # 2. Gradiente gaussiano inverso
    axes[1].imshow(gradiente_inverso, cmap="magma")
    axes[1].set_title("2. Gradiente gaussiano inverso", fontsize=11, pad=8)
    axes[1].axis("off")

    # 3. Inicialización circular
    axes[2].imshow(ventana_pre, cmap="gray", vmin=0, vmax=1)
    axes[2].contour(inicial, levels=[0.5], colors=[COLOR_INIT], linewidths=2)
    axes[2].set_title(f"3. Inicialización\nradio={radio_ini} px", fontsize=11, pad=8)
    axes[2].axis("off")

    # 4. Máscara GAC cruda
    axes[3].imshow(gac_crudo, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title("4. Máscara GAC cruda", fontsize=11, pad=8)
    axes[3].axis("off")

    # 5. Máscara GAC final
    axes[4].imshow(gac_final, cmap="gray", vmin=0, vmax=1)
    axes[4].set_title("5. Máscara GAC final", fontsize=11, pad=8)
    axes[4].axis("off")

    fig.subplots_adjust(
        top=0.82,
        bottom=0.04,
        left=0.02,
        right=0.98,
        wspace=0.08
    )

    plt.savefig(
        FIG_DIR / f"etapas_gac_{idx:03d}_{caso['case_id']}.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    filas_gac_visual.append({
        "idx": idx,
        "case_id": caso["case_id"],
        "patient_id": caso["patient_id"],
        "vista": caso["vista"],
        "radio_inicial_px": radio_ini,
        "num_iter": 105,
        "smoothing": 1,
        "balloon": 1.0,
        "alpha_gradiente": 60,
        "sigma_gradiente": 1.5,
        "area_gac_cruda_px": int(gac_crudo.sum()),
        "area_gac_final_px": int(gac_final.sum()),
        "borde_corregido": borde_corregido
    })


df_gac_visual = pd.DataFrame(filas_gac_visual)

print("Parámetros y áreas de GAC en los casos visualizados:")
display(df_gac_visual.reset_index(drop=True))

df_gac_visual.to_csv(
    OUT_DIR / "visualizacion_parametros_gac.csv",
    index=False
)

### Visualización de K-means

Esta celda muestra las etapas principales de la segmentación por K-means. Se usa la ventana preprocesada y una versión suavizada. Luego cada píxel se representa con dos variables: su intensidad y su cercanía al centro de la ventana.



In [ ]:
# @title
# ============================================================
# Visualización - Segmentación por K-means
# ============================================================

from sklearn.cluster import KMeans
from matplotlib.colors import ListedColormap

IDX_VIS_KMEANS = [4, 5, 8, 16, 27, 2]
IDX_VIS_KMEANS = [i for i in IDX_VIS_KMEANS if i < len(df_casos_batch)]

# Colores suaves
COLOR_GT = "#7E9F90"
COLOR_KM = "#A8C686"

cmap_labels = ListedColormap([
    "#D8DEE9",  # gris azulado suave
    "#B7D7E8",  # celeste suave
    "#C9DCA9"   # verde suave
])

filas_scores_kmeans = []

for idx in IDX_VIS_KMEANS:

    caso = df_casos_batch.iloc[idx]

    ventana, gt, info_ventana = preparar_ventana_y_gt(caso)
    ventana_pre, params_pre = preprocesar_miniroi_log_clahe(ventana)

    img = np.clip(ventana_pre.astype(np.float32), 0, 1)

    h, w = img.shape
    yy, xx = np.mgrid[:h, :w]
    cy, cx = h // 2, w // 2

    img_suave = filters.gaussian(img, sigma=1.2)

    dist = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
    dist = dist / (dist.max() + 1e-8)
    cercania_centro = 1 - dist

    peso_distancia = 0.10

    datos = np.column_stack([
        img_suave.ravel(),
        peso_distancia * cercania_centro.ravel()
    ])

    kmeans = KMeans(
        n_clusters=3,
        random_state=0,
        n_init=10
    )

    labels = kmeans.fit_predict(datos).reshape(h, w)

    scores = []

    for lab in range(3):
        region = labels == lab

        if region.sum() == 0:
            score = -np.inf
            intensidad_media = np.nan
            cercania_media = np.nan
            area = 0
        else:
            intensidad_media = img_suave[region].mean()
            cercania_media = cercania_centro[region].mean()
            score = intensidad_media + peso_distancia * cercania_media
            area = int(region.sum())

        scores.append(score)

        filas_scores_kmeans.append({
            "idx": idx,
            "case_id": caso["case_id"],
            "cluster": lab,
            "area_px": area,
            "intensidad_media": intensidad_media,
            "cercania_media": cercania_media,
            "score": score
        })

    cluster_masa = int(np.argmax(scores))
    seg_kmeans = limpiar_mask(labels == cluster_masa)

    # ------------------------------------------------------------
    # Figura
    # ------------------------------------------------------------
    fig, axes = plt.subplots(
        1,
        4,
        figsize=(18, 4.8),
        constrained_layout=False
    )

    fig.suptitle(
        f"K-means — {caso['patient_id']} | {caso['lateralidad']} | {caso['vista']}",
        fontsize=14,
        y=0.98
    )

    # 1. Ventana preprocesada
    axes[0].imshow(ventana_pre, cmap="gray", vmin=0, vmax=1)
    axes[0].contour(gt, levels=[0.5], colors=[COLOR_GT], linewidths=1.6)
    axes[0].set_title("1. Ventana preprocesada", fontsize=11, pad=8)
    axes[0].axis("off")

    # 2. Imagen suavizada
    axes[1].imshow(img_suave, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("2. Suavizado gaussiano", fontsize=11, pad=8)
    axes[1].axis("off")

    # 3. Clusters
    axes[2].imshow(labels, cmap=cmap_labels, vmin=0, vmax=2)
    axes[2].set_title(f"3. Clusters K=3\nseleccionado: {cluster_masa}", fontsize=11, pad=8)
    axes[2].axis("off")

    # 4. Máscara final
    axes[3].imshow(seg_kmeans, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title("4. Máscara K-means", fontsize=11, pad=8)
    axes[3].axis("off")

    fig.subplots_adjust(
        top=0.80,
        bottom=0.04,
        left=0.02,
        right=0.98,
        wspace=0.08
    )

    plt.savefig(
        FIG_DIR / f"etapas_kmeans_{idx:03d}_{caso['case_id']}.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()


df_scores_kmeans = pd.DataFrame(filas_scores_kmeans)

print("Scores de K-means para los casos visualizados:")
display(
    df_scores_kmeans.round(4).reset_index(drop=True)
)

df_scores_kmeans.to_csv(
    OUT_DIR / "scores_kmeans_casos_visualizados.csv",
    index=False
)

### Visualización del método híbrido

Esta celda muestra cómo se combinan las máscaras binarias obtenidas por GAC y K-means. La máscara de GAC se toma como base. Luego se genera una banda alrededor de esa máscara y se permite agregar únicamente los píxeles detectados por K-means que caen dentro de esa zona cercana.

In [ ]:
# @title
# ============================================================
# Visualización - Método híbrido GAC + K-means en binario
# ============================================================

IDX_VIS_HIBRIDO = [4, 5, 8, 16, 27, 30]
IDX_VIS_HIBRIDO = [i for i in IDX_VIS_HIBRIDO if i < len(df_casos_batch)]

filas_hibrido = []

for idx in IDX_VIS_HIBRIDO:

    caso = df_casos_batch.iloc[idx]

    ventana, gt, info_ventana = preparar_ventana_y_gt(caso)
    ventana_pre, params_pre = preprocesar_miniroi_log_clahe(ventana)

    seg_gac, seg_kmeans, seg_hibrida, info_gac = segmentar_pipeline(
        ventana_pre,
        devolver_info=True
    )

    radio_banda = globals().get("RADIO_BANDA_HIBRIDO", 6)

    zona_cercana_gac = morphology.binary_dilation(
        seg_gac,
        morphology.disk(radio_banda)
    )

    aporte_kmeans = seg_kmeans & zona_cercana_gac & (~seg_gac)

    # ------------------------------------------------------------
    # Figura binaria
    # ------------------------------------------------------------
    fig, axes = plt.subplots(
        1,
        5,
        figsize=(20, 4.8),
        constrained_layout=False
    )

    fig.suptitle(
        f"Método híbrido GAC + K-means — {caso['patient_id']} | {caso['lateralidad']} | {caso['vista']}",
        fontsize=14,
        y=0.98
    )

    axes[0].imshow(seg_gac, cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("1. Máscara GAC", fontsize=11, pad=8)
    axes[0].axis("off")

    axes[1].imshow(seg_kmeans, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("2. Máscara K-means", fontsize=11, pad=8)
    axes[1].axis("off")

    axes[2].imshow(zona_cercana_gac, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title(f"3. Banda cercana a GAC\nradio={radio_banda} px", fontsize=11, pad=8)
    axes[2].axis("off")

    axes[3].imshow(aporte_kmeans, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title("4. Aporte aceptado\nde K-means", fontsize=11, pad=8)
    axes[3].axis("off")

    axes[4].imshow(seg_hibrida, cmap="gray", vmin=0, vmax=1)
    axes[4].set_title("5. Máscara híbrida final", fontsize=11, pad=8)
    axes[4].axis("off")

    fig.subplots_adjust(
        top=0.80,
        bottom=0.04,
        left=0.02,
        right=0.98,
        wspace=0.08
    )

    plt.savefig(
        FIG_DIR / f"etapas_hibrido_binario_{idx:03d}_{caso['case_id']}.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    filas_hibrido.append({
        "idx": idx,
        "case_id": caso["case_id"],
        "patient_id": caso["patient_id"],
        "lateralidad": caso["lateralidad"],
        "vista": caso["vista"],
        "area_gac_px": int(seg_gac.sum()),
        "area_kmeans_px": int(seg_kmeans.sum()),
        "area_banda_px": int(zona_cercana_gac.sum()),
        "aporte_kmeans_px": int(aporte_kmeans.sum()),
        "area_hibrida_px": int(seg_hibrida.sum()),
        "radio_banda_hibrido": radio_banda,
        "puente_gac_aplicado": info_gac.get("puente_aplicado", False),
        "borde_gac_corregido": info_gac.get("borde_corregido", False)
    })


df_hibrido_visual = pd.DataFrame(filas_hibrido)

print("Resumen de áreas del método híbrido:")
display(df_hibrido_visual.reset_index(drop=True))

df_hibrido_visual.to_csv(
    OUT_DIR / "resumen_visual_hibrido.csv",
    index=False
)

### Comparación visual de las máscaras finales

En esta celda se comparan las salidas finales de los tres métodos de segmentación. Para cada caso se muestra la ventana preprocesada, la máscara de referencia y las máscaras binarias obtenidas con GAC, K-means y el método híbrido.


In [ ]:
# @title
# ============================================================
# Comparación visual de máscaras finales
# ============================================================

IDX_VIS_COMPARACION = [4, 5, 8, 2, 25, 30]
IDX_VIS_COMPARACION = [i for i in IDX_VIS_COMPARACION if i < len(df_casos_batch)]

COLOR_GT = "#7E9F90"       # verde grisáceo suave
COLOR_GAC = "#8FBBD9"      # celeste suave
COLOR_KM = "#A8C686"       # verde suave
COLOR_HIB = "#B8A9C9"      # lila suave

filas_comparacion_visual = []

for idx in IDX_VIS_COMPARACION:

    caso = df_casos_batch.iloc[idx]

    # ------------------------------------------------------------
    # Preparación de ventana y segmentación
    # ------------------------------------------------------------
    ventana, gt, info_ventana = preparar_ventana_y_gt(caso)
    ventana_pre, params_pre = preprocesar_miniroi_log_clahe(ventana)

    seg_gac, seg_kmeans, seg_hibrida, info_seg = segmentar_pipeline(
        ventana_pre,
        devolver_info=True
    )

    # ------------------------------------------------------------
    # Figura comparativa
    # ------------------------------------------------------------
    fig, axes = plt.subplots(
        1,
        5,
        figsize=(20, 4.8),
        constrained_layout=False
    )

    fig.suptitle(
        f"Comparación de máscaras finales — {caso['patient_id']} | {caso['lateralidad']} | {caso['vista']}",
        fontsize=14,
        y=0.98
    )

    # 1. Ventana preprocesada + GT
    axes[0].imshow(ventana_pre, cmap="gray", vmin=0, vmax=1)
    axes[0].contour(gt, levels=[0.5], colors=[COLOR_GT], linewidths=1.8)
    axes[0].set_title("1. Ventana preprocesada\n+ referencia", fontsize=11, pad=8)
    axes[0].axis("off")

    # 2. GT binaria
    axes[1].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("2. Máscara de referencia", fontsize=11, pad=8)
    axes[1].axis("off")

    # 3. GAC binaria
    axes[2].imshow(seg_gac, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("3. Máscara GAC", fontsize=11, pad=8)
    axes[2].axis("off")

    # 4. K-means binaria
    axes[3].imshow(seg_kmeans, cmap="gray", vmin=0, vmax=1)
    axes[3].set_title("4. Máscara K-means", fontsize=11, pad=8)
    axes[3].axis("off")

    # 5. Híbrida binaria
    axes[4].imshow(seg_hibrida, cmap="gray", vmin=0, vmax=1)
    axes[4].set_title("5. Máscara híbrida", fontsize=11, pad=8)
    axes[4].axis("off")

    fig.subplots_adjust(
        top=0.80,
        bottom=0.04,
        left=0.02,
        right=0.98,
        wspace=0.08
    )

    plt.savefig(
        FIG_DIR / f"comparacion_mascaras_finales_{idx:03d}_{caso['case_id']}.png",
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()

    # ------------------------------------------------------------
    # Resumen de áreas
    # ------------------------------------------------------------
    filas_comparacion_visual.append({
        "idx": idx,
        "case_id": caso["case_id"],
        "patient_id": caso["patient_id"],
        "lateralidad": caso["lateralidad"],
        "vista": caso["vista"],
        "area_gt_px": int(gt.sum()),
        "area_gac_px": int(seg_gac.sum()),
        "area_kmeans_px": int(seg_kmeans.sum()),
        "area_hibrida_px": int(seg_hibrida.sum()),
        "borde_gac_corregido": info_seg.get("borde_corregido", False),
        "puente_gac_aplicado": info_seg.get("puente_aplicado", False)
    })


df_comparacion_visual = pd.DataFrame(filas_comparacion_visual)

print("Resumen de áreas de las máscaras visualizadas:")
display(df_comparacion_visual.reset_index(drop=True))

df_comparacion_visual.to_csv(
    OUT_DIR / "comparacion_visual_areas_mascaras.csv",
    index=False
)

print("\nTabla guardada en:")
print(OUT_DIR / "comparacion_visual_areas_mascaras.csv")

# *4. Métricas de Segmentación*

Esta celda define la función `metricas_segmentacion(seg, gt)`, que compara una máscara segmentada (`seg`) con la máscara de referencia (`gt`). Ambas máscaras se convierten a booleanas, de modo que cada píxel queda clasificado como perteneciente o no perteneciente a la masa.

La función calcula primero los conteos píxel a píxel: verdaderos positivos (`TP`), falsos positivos (`FP`) y falsos negativos (`FN`). A partir de esos valores obtiene las métricas principales de segmentación:

$$
Dice = \frac{2TP}{2TP + FP + FN}
$$

$$
Jaccard = \frac{TP}{TP + FP + FN}
$$

$$
Precisión = \frac{TP}{TP + FP}
$$

$$
Sensibilidad = \frac{TP}{TP + FN}
$$

Además, calcula el área de la máscara segmentada (`area_seg`), el área de la máscara de referencia (`area_gt`) y el cociente entre ambas:

$$
Ratio_area = \frac{area_seg}{area_gt}
$$

Este último valor se usa como criterio simple para clasificar si la segmentación tiende a subsegmentar, sobresegmentar o mantener un área razonable respecto de la referencia.


In [ ]:
# ============================================================
# CELDA 7 - Métricas de segmentación
# ============================================================

def metricas_segmentacion(seg, gt):
    seg = seg.astype(bool)
    gt = gt.astype(bool)

    tp = np.logical_and(seg, gt).sum()
    fp = np.logical_and(seg, ~gt).sum()
    fn = np.logical_and(~seg, gt).sum()

    area_seg = seg.sum()
    area_gt = gt.sum()

    dice = (2 * tp) / (2 * tp + fp + fn + 1e-8)
    jaccard = tp / (tp + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    sensibilidad = tp / (tp + fn + 1e-8)

    ratio_area = area_seg / (area_gt + 1e-8)

    if ratio_area < 0.70:
        diagnostico_area = "subsegmenta"
    elif ratio_area > 1.50:
        diagnostico_area = "sobresegmenta"
    else:
        diagnostico_area = "area_razonable"

    return {
        "dice": dice,
        "jaccard": jaccard,
        "precision": precision,
        "sensibilidad": sensibilidad,
        "area_seg": area_seg,
        "area_gt": area_gt,
        "ratio_area": ratio_area,
        "diagnostico_area": diagnostico_area
    }


# *5. Extracción de características*

Esta celda define funciones para extraer características de una máscara binaria.

Las características simples que se utilizan para el análisis son: `area_px`, `perimetro_px`, `circularidad`, `irregularidad_borde`, `solidez`, `intensidad_media` e `intensidad_std`.

`area_px` se calcula como la cantidad de píxeles pertenecientes a la máscara. `perimetro_px` estima la longitud del borde de la región. A partir de ambas se calcula la circularidad:

$$
Circularidad = \frac{4 \pi \cdot Area}{Perimetro^2}
$$

La circularidad toma valores más altos cuando la región es más compacta y parecida a un círculo. Como medida complementaria se calcula `irregularidad_borde`:

$$
Irregularidad = \frac{Perimetro^2}{4 \pi \cdot Area}
$$

Esta aumenta cuando el perímetro es grande en relación con el área, es decir, cuando el borde es más irregular.

La `solidez` compara el área de la región con el área de su envolvente convexa. Valores cercanos a 1 indican regiones más compactas, mientras que valores menores pueden asociarse a formas con concavidades o bordes más irregulares.

Además, se calculan `intensidad_media` e `intensidad_std` dentro de la máscara.

In [ ]:
# ============================================================
# CELDA 8 - Características simples de la máscara
# Forma + borde + intensidad
# ============================================================

def perimetro_seguro(mask):
    """
    Calcula el perímetro de una máscara binaria.
    """
    mask = mask.astype(bool)

    try:
        return measure.perimeter(mask, neighborhood=8)
    except TypeError:
        try:
            return measure.perimeter(mask, neighbourhood=8)
        except TypeError:
            return measure.perimeter(mask)


def caracteristicas_forma(mask, imagen=None, prefijo=""):
    """
    Extrae características seleccionadas de una máscara binaria.

    mask:
        Máscara binaria de la masa.

    imagen:
        Imagen de intensidades alineada con la máscara.
        En este pipeline se usa la ventana normalizada/preprocesada,
        del mismo tamaño que gt y seg_hibrida.

    prefijo:
        'gt_' o 'seg_' para distinguir máscara de referencia y máscara segmentada.
    """

    mask = mask.astype(bool)

    columnas_nan = {
        f"{prefijo}area_px": np.nan,
        f"{prefijo}perimetro_px": np.nan,
        f"{prefijo}circularidad": np.nan,
        f"{prefijo}irregularidad_borde": np.nan,
        f"{prefijo}solidez": np.nan,
        f"{prefijo}intensidad_media": np.nan,
        f"{prefijo}intensidad_std": np.nan
    }

    if mask.sum() == 0:
        return columnas_nan

    labels = measure.label(mask)
    props = measure.regionprops(labels)

    if len(props) == 0:
        return columnas_nan

    # Si hubiera más de una componente, tomo la de mayor área.
    p = max(props, key=lambda x: x.area)

    # Máscara sólo de la componente principal.
    mask_obj = labels == p.label

    # ------------------------------------------------------------
    # Características geométricas
    # ------------------------------------------------------------
    area = float(p.area)
    perimetro = float(perimetro_seguro(mask_obj))

    circularidad = (4 * np.pi * area) / (perimetro**2 + 1e-8)

    irregularidad_borde = (perimetro**2) / (4 * np.pi * area + 1e-8)

    solidez = float(p.solidity)

    # ------------------------------------------------------------
    # Características de intensidad
    # ------------------------------------------------------------
    if imagen is not None and imagen.shape == mask_obj.shape:
        valores = imagen[mask_obj]

        if len(valores) > 0:
            intensidad_media = float(np.mean(valores))
            intensidad_std = float(np.std(valores))
        else:
            intensidad_media = np.nan
            intensidad_std = np.nan
    else:
        intensidad_media = np.nan
        intensidad_std = np.nan

    return {
        f"{prefijo}area_px": area,
        f"{prefijo}perimetro_px": perimetro,
        f"{prefijo}circularidad": circularidad,
        f"{prefijo}irregularidad_borde": irregularidad_borde,
        f"{prefijo}solidez": solidez,
        f"{prefijo}intensidad_media": intensidad_media,
        f"{prefijo}intensidad_std": intensidad_std
    }

# *6. Procesamiento completo*

Esta celda ejecuta el pipeline completo para todos los casos seleccionados. Para cada fila de `df_entrada`, primero se construye la ventana local con `preparar_ventana_y_gt`, después se aplica el preprocesamiento adaptativo con `preprocesar_miniroi_log_clahe` y luego se segmenta la masa con `segmentar_pipeline`.

La función `segmentar_pipeline` devuelve tres máscaras: `seg_gac`, `seg_kmeans` y `seg_hibrida`. Para cada una se calculan métricas con `metricas_segmentacion`. La máscara híbrida se toma como segmentación principal para la extracción de características, por eso `df_resultados` guarda las métricas y características asociadas al método híbrido.

Además, la celda guarda una segunda tabla, `df_metodos`, donde quedan las métricas de los tres métodos por separado. Esta tabla se usa después para comparar GAC, K-means e híbrido.

También se guardan los archivos `resultados_pipeline_final.csv`, `resultados_metodos_pipeline_final.csv` y `errores_pipeline_final.csv`.


In [ ]:
# ============================================================
# CELDA 9 - Batch principal con limpieza de RAM y guardado de figuras
# ============================================================

PROCESAR_TODO = True
GUARDAR_FIGURAS = True
GUARDAR_TODAS_LAS_FIGURAS = True
DPI_FIGURAS = 120

df_entrada = df_casos_ok.copy() if PROCESAR_TODO else df_casos_batch.copy()
df_entrada = df_entrada.reset_index(drop=True)

resultados = []
resultados_metodos = []
errores = []

FIG_DIR.mkdir(parents=True, exist_ok=True)

# Limpio figuras anteriores para no mezclar resultados viejos con resultados nuevos.
if GUARDAR_FIGURAS:
    for f in FIG_DIR.glob("caso_*.png"):
        f.unlink()


def valor(caso, col):
    if col in caso.index:
        return caso[col]
    return np.nan

def agregar_metricas_mm(fila, prefijo, spacing_y, spacing_x):
    """
    Convierte a unidades físicas sólo las características seleccionadas
    que dependen de tamaño: área y perímetro.

    Como en caracteristicas_forma ya no se calculan diámetro equivalente
    ni ejes principales, esas conversiones se eliminan.
    """

    fila[f"{prefijo}area_mm2"] = np.nan
    fila[f"{prefijo}perimetro_mm"] = np.nan

    if pd.isna(spacing_y) or pd.isna(spacing_x):
        return fila

    try:
        spacing_y = float(spacing_y)
        spacing_x = float(spacing_x)

        # Para área se usan ambos espaciamientos.
        fila[f"{prefijo}area_mm2"] = fila[f"{prefijo}area_px"] * spacing_y * spacing_x

        # Para perímetro se usa un espaciamiento promedio como aproximación.
        spacing_prom = (spacing_y + spacing_x) / 2
        fila[f"{prefijo}perimetro_mm"] = fila[f"{prefijo}perimetro_px"] * spacing_prom

    except Exception:
        fila[f"{prefijo}area_mm2"] = np.nan
        fila[f"{prefijo}perimetro_mm"] = np.nan

    return fila


def guardar_figura_caso(i, caso, ventana, ventana_pre, gt, seg_gac, seg_kmeans, seg_hibrida, met_seg):
    nombre = (
        f"caso_{i:03d}_"
        f"{caso['patient_id']}_"
        f"{caso['lateralidad']}_"
        f"{caso['vista']}.png"
    )

    fig, ax = plt.subplots(1, 6, figsize=(20, 4))

    ax[0].imshow(ventana, cmap="gray")
    ax[0].set_title("Ventana base")
    ax[0].axis("off")

    ax[1].imshow(ventana_pre, cmap="gray")
    ax[1].set_title("Preprocesada")
    ax[1].axis("off")

    ax[2].imshow(ventana, cmap="gray")
    ax[2].imshow(gt, cmap="Reds", alpha=0.35)
    ax[2].set_title("GT")
    ax[2].axis("off")

    ax[3].imshow(ventana_pre, cmap="gray")
    ax[3].imshow(seg_gac, cmap="Blues", alpha=0.35)
    ax[3].set_title("GAC")
    ax[3].axis("off")

    ax[4].imshow(ventana_pre, cmap="gray")
    ax[4].imshow(seg_kmeans, cmap="Greens", alpha=0.35)
    ax[4].set_title("K-means")
    ax[4].axis("off")

    ax[5].imshow(ventana_pre, cmap="gray")
    ax[5].imshow(seg_hibrida, cmap="Purples", alpha=0.35)
    ax[5].set_title(f"Híbrido\nDice={met_seg['dice']:.3f}")
    ax[5].axis("off")

    titulo = (
        f"{caso['patient_id']} | {caso['lateralidad']} | {caso['vista']} | "
        f"{caso['mass_shape']} | {caso['mass_margins']}"
    )

    fig.suptitle(titulo, fontsize=10)
    plt.tight_layout()

    fig.savefig(FIG_DIR / nombre, dpi=DPI_FIGURAS, bbox_inches="tight")
    plt.close(fig)


for i, (_, caso) in enumerate(df_entrada.iterrows()):

    try:
        print(
            f"Procesando caso {i + 1}/{len(df_entrada)}: "
            f"{caso['patient_id']} {caso['lateralidad']} {caso['vista']}"
        )

        ventana, gt, info = preparar_ventana_y_gt(caso)
        ventana_pre, params_pre = preprocesar_miniroi_log_clahe(ventana)

        seg_gac, seg_kmeans, seg_hibrida, info_gac = segmentar_pipeline(
            ventana_pre,
            devolver_info=True
        )

        segmentaciones = {
            "GAC": seg_gac,
            "Kmeans": seg_kmeans,
            "Hibrido": seg_hibrida
        }

        metricas_por_metodo = {}

        # Se calculan métricas para los tres métodos sin volver a segmentar.
        for metodo, seg_metodo in segmentaciones.items():
            met = metricas_segmentacion(seg_metodo, gt)
            metricas_por_metodo[metodo] = met

            resultados_metodos.append({
                "case_id": valor(caso, "case_id"),
                "patient_id": valor(caso, "patient_id"),
                "lateralidad": valor(caso, "lateralidad"),
                "vista": valor(caso, "vista"),
                "breast_density": valor(caso, "breast_density"),
                "mass_shape": valor(caso, "mass_shape"),
                "mass_margins": valor(caso, "mass_margins"),
                "pathology": valor(caso, "pathology"),
                "subtlety": valor(caso, "subtlety"),
                "metodo": metodo,
                "condicion": "pre",
                **met
            })

        met_seg = metricas_por_metodo["Hibrido"]

        car_gt = caracteristicas_forma(gt, imagen=ventana, prefijo="gt_")
        car_seg = caracteristicas_forma(seg_hibrida, imagen=ventana, prefijo="seg_")

        fila = {
            "case_id": valor(caso, "case_id"),
            "patient_id": valor(caso, "patient_id"),
            "lateralidad": valor(caso, "lateralidad"),
            "vista": valor(caso, "vista"),

            "breast_density": valor(caso, "breast_density"),
            "mass_shape": valor(caso, "mass_shape"),
            "mass_margins": valor(caso, "mass_margins"),
            "assessment": valor(caso, "assessment"),
            "pathology": valor(caso, "pathology"),
            "subtlety": valor(caso, "subtlety"),

            "ruta_mg": valor(caso, "ruta_mg"),
            "ruta_roi": valor(caso, "ruta_roi"),
            "ruta_mask": valor(caso, "ruta_mask"),

            "shape_ventana": str(ventana.shape),
            "shape_gt": str(gt.shape),

            "radio_banda_hibrido": RADIO_BANDA_HIBRIDO,

            "gain_log": params_pre["gain_log"],
            "clip_limit": params_pre["clip_limit"],
            "kernel_size": params_pre["kernel_size"],
            "gain_log_usado": params_pre["gain_log_usado"],
            "clip_limit_usado": params_pre["clip_limit_usado"],
            "kernel_size_usado": params_pre["kernel_size_usado"],
            "separacion_miniroi": params_pre["separacion"],
            "competencia_fondo": params_pre["competencia_fondo"],
            "saturacion_centro": params_pre["saturacion_centro"],
            "frac_centro_quemado_post": params_pre["frac_centro_quemado_post"],
            "anti_saturacion": params_pre["anti_saturacion"],

            "puente_aplicado": info_gac["puente_aplicado"],
            "pixeles_puente_agregados": info_gac["pixeles_puente_agregados"],
            "borde_corregido": info_gac["borde_corregido"],
            "toca_arriba": info_gac["toca_arriba"],
            "toca_abajo": info_gac["toca_abajo"],
            "toca_izquierda": info_gac["toca_izquierda"],
            "toca_derecha": info_gac["toca_derecha"]
        }

        fila.update(met_seg)
        fila.update(car_gt)
        fila.update(car_seg)

        fila = agregar_metricas_mm(
            fila,
            "gt_",
            valor(caso, "spacing_y_mm"),
            valor(caso, "spacing_x_mm")
        )

        fila = agregar_metricas_mm(
            fila,
            "seg_",
            valor(caso, "spacing_y_mm"),
            valor(caso, "spacing_x_mm")
        )

        resultados.append(fila)

        if GUARDAR_FIGURAS and GUARDAR_TODAS_LAS_FIGURAS:
            guardar_figura_caso(
                i,
                caso,
                ventana,
                ventana_pre,
                gt,
                seg_gac,
                seg_kmeans,
                seg_hibrida,
                met_seg
            )

    except Exception as e:
        errores.append({
            "case_id": valor(caso, "case_id"),
            "patient_id": valor(caso, "patient_id"),
            "lateralidad": valor(caso, "lateralidad"),
            "vista": valor(caso, "vista"),
            "error": str(e)
        })

    finally:
        # Limpieza de RAM: borro matrices grandes y cierro figuras.
        for nombre_var in [
            "ventana", "ventana_pre", "gt",
            "seg_gac", "seg_kmeans", "seg_hibrida",
            "segmentaciones", "metricas_por_metodo",
            "met_seg", "car_gt", "car_seg", "info", "params_pre", "info_gac"
        ]:
            if nombre_var in locals():
                del locals()[nombre_var]

        plt.close("all")
        gc.collect()


df_resultados = pd.DataFrame(resultados)
df_metodos = pd.DataFrame(resultados_metodos)
df_errores = pd.DataFrame(errores)

df_resultados.to_csv(OUT_DIR / "resultados_pipeline_final.csv", index=False)
df_metodos.to_csv(OUT_DIR / "resultados_metodos_pipeline_final.csv", index=False)
df_errores.to_csv(OUT_DIR / "errores_pipeline_final.csv", index=False)

print("\nBatch terminado.")
print("Casos a procesar:", len(df_entrada))
print("Casos procesados:", len(df_resultados))
print("Casos con error:", len(df_errores))
print("Resultados híbridos guardados en:", OUT_DIR / "resultados_pipeline_final.csv")
print("Comparación de métodos guardada en:", OUT_DIR / "resultados_metodos_pipeline_final.csv")
print("Errores guardados en:", OUT_DIR / "errores_pipeline_final.csv")
print("Figuras guardadas en:", FIG_DIR)

cols_preview = [
    "case_id",
    "patient_id",
    "lateralidad",
    "vista",
    "mass_shape",
    "mass_margins",
    "dice",
    "jaccard",
    "precision",
    "sensibilidad",
    "ratio_area",
    "diagnostico_area",
    "gt_area_px",
    "seg_area_px",
    "gt_circularidad",
    "seg_circularidad",
    "gt_irregularidad_borde",
    "seg_irregularidad_borde",
    "gt_solidez",
    "seg_solidez",
    "gt_intensidad_media",
    "seg_intensidad_media",
    "anti_saturacion",
    "borde_corregido"
]

cols_preview = [c for c in cols_preview if c in df_resultados.columns]

display(df_resultados[cols_preview].head())

if len(df_errores) > 0:
    print("\nErrores:")
    display(df_errores)


### Figuras de control del pipeline completo

Las siguientes imágenes muestran algunos casos procesados por el batch completo.

In [ ]:
# @title
# ============================================================
# CELDA EXTRA - Visualización de figuras de control del batch
# ============================================================

from PIL import Image

# Elegir manualmente qué figuras mostrar.
# Son los índices del batch, no los patient_id.
IDX_FIGURAS_CONTROL = [4, 5, 8, 2, 25, 30]

rutas_figuras = sorted(FIG_DIR.glob("caso_*.png"))

rutas_mostrar = [
    rutas_figuras[i]
    for i in IDX_FIGURAS_CONTROL
    if i < len(rutas_figuras)
]

print("Figuras de control seleccionadas:")
for r in rutas_mostrar:
    print(r.name)

for ruta in rutas_mostrar:
    img = Image.open(ruta)

    fig, ax = plt.subplots(figsize=(18, 4.2))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(ruta.name, fontsize=11, pad=8)

    plt.tight_layout()
    plt.show()

# *7. Resultados de la segmentación y características simples*

A partir del batch completo se construyen tablas de resultados para analizar el desempeño del pipeline. En esta parte ya no se vuelve a segmentar: se usan los resultados guardados en `df_resultados` y `df_metodos`.

La tabla `df_resultados` contiene una fila por caso y resume el resultado del método híbrido, que se toma como salida principal del pipeline. Incluye las métricas de segmentación, los parámetros usados en el preprocesamiento y las características calculadas sobre la máscara de referencia y sobre la máscara segmentada.

En esta celda se calculan diferencias simples entre las características de la segmentación y las de la referencia. Esto permite ver si la máscara segmentada tiende a aumentar o disminuir el área, el perímetro, la circularidad, la irregularidad, la solidez o las intensidades respecto de la máscara original.


In [ ]:
# @title
# ============================================================
# CELDA 10 - Caracterización GT vs segmentación
# ============================================================

# Si se cerró Colab y se quieren cargar resultados ya guardados:
# df_resultados = pd.read_csv(OUT_DIR / "resultados_pipeline_final.csv")

df_caracterizacion = df_resultados.copy()

# ------------------------------------------------------------
# Aseguro columna case_id
# ------------------------------------------------------------
if "case_id" not in df_caracterizacion.columns:
    df_caracterizacion["case_id"] = (
        df_caracterizacion["patient_id"].astype(str) + "_" +
        df_caracterizacion["lateralidad"].astype(str) + "_" +
        df_caracterizacion["vista"].astype(str)
    )

# ------------------------------------------------------------
# Características seleccionadas para el análisis final
# ------------------------------------------------------------
caracteristicas_seleccionadas = [
    "area_px",
    "perimetro_px",
    "circularidad",
    "irregularidad_borde",
    "solidez",
    "intensidad_media",
    "intensidad_std"
]

# Me quedo sólo con las características que efectivamente existan
# tanto para GT como para segmentación.
caracteristicas_seleccionadas = [
    f for f in caracteristicas_seleccionadas
    if f"gt_{f}" in df_caracterizacion.columns
    and f"seg_{f}" in df_caracterizacion.columns
]

# ------------------------------------------------------------
# Diferencias entre segmentación y referencia
# ------------------------------------------------------------
for f in caracteristicas_seleccionadas:
    df_caracterizacion[f"delta_{f}"] = (
        df_caracterizacion[f"seg_{f}"] - df_caracterizacion[f"gt_{f}"]
    )

# ------------------------------------------------------------
# Interpretaciones simples de forma
# ------------------------------------------------------------
if "circularidad" in caracteristicas_seleccionadas:
    df_caracterizacion["seg_mas_redondeada_que_gt"] = (
        df_caracterizacion["seg_circularidad"] > df_caracterizacion["gt_circularidad"]
    )

if "irregularidad_borde" in caracteristicas_seleccionadas:
    df_caracterizacion["seg_menos_irregular_que_gt"] = (
        df_caracterizacion["seg_irregularidad_borde"] <
        df_caracterizacion["gt_irregularidad_borde"]
    )

if "solidez" in caracteristicas_seleccionadas:
    df_caracterizacion["seg_mas_solida_que_gt"] = (
        df_caracterizacion["seg_solidez"] > df_caracterizacion["gt_solidez"]
    )

# ------------------------------------------------------------
# Marca auxiliar para identificar casos que conviene revisar visualmente
# ------------------------------------------------------------
df_caracterizacion["revisar_segmentacion"] = (
    (df_caracterizacion["dice"] < 0.70) |
    (df_caracterizacion["precision"] < 0.70) |
    (df_caracterizacion["sensibilidad"] < 0.70) |
    (df_caracterizacion["ratio_area"] < 0.70) |
    (df_caracterizacion["ratio_area"] > 1.50)
)

# ------------------------------------------------------------
# Clasificación relativa dentro de la muestra: bajo / medio / alto
# ------------------------------------------------------------
def categoria_relativa(serie):
    q25 = serie.quantile(0.25)
    q75 = serie.quantile(0.75)

    def clasificar(x):
        if pd.isna(x):
            return np.nan
        if x <= q25:
            return "bajo"
        elif x >= q75:
            return "alto"
        else:
            return "medio"

    return serie.apply(clasificar)


# Sólo categorizo características seleccionadas que se van a usar o interpretar.
for col in [
    "gt_circularidad",
    "gt_irregularidad_borde",
    "gt_solidez",
    "gt_intensidad_media",
    "gt_intensidad_std"
]:
    if col in df_caracterizacion.columns:
        df_caracterizacion[col + "_cat"] = categoria_relativa(df_caracterizacion[col])

# ------------------------------------------------------------
# Guardado de tabla completa
# ------------------------------------------------------------
df_caracterizacion.to_csv(
    OUT_DIR / "resultados_caracterizados.csv",
    index=False
)

# ------------------------------------------------------------
# Salida resumida para el notebook
# ------------------------------------------------------------
print("Casos caracterizados:", len(df_caracterizacion))
print("Archivo completo guardado en:")
print(OUT_DIR / "resultados_caracterizados.csv")

print("\nResumen de casos a revisar:")
display(
    df_caracterizacion["revisar_segmentacion"]
    .value_counts(dropna=False)
    .rename_axis("revisar_segmentacion")
    .reset_index(name="cantidad")
)

print("\nVista previa de casos caracterizados:")

cols_preview = [
    "case_id",
    "mass_shape",
    "mass_margins",
    "dice",
    "precision",
    "sensibilidad",
    "ratio_area",
    "gt_area_px",
    "seg_area_px",
    "delta_area_px",
    "gt_circularidad",
    "seg_circularidad",
    "delta_circularidad",
    "gt_irregularidad_borde",
    "seg_irregularidad_borde",
    "delta_irregularidad_borde",
    "gt_solidez",
    "seg_solidez",
    "delta_solidez",
    "gt_intensidad_media",
    "seg_intensidad_media",
    "delta_intensidad_media",
    "revisar_segmentacion"
]

cols_preview = [c for c in cols_preview if c in df_caracterizacion.columns]

display(
    df_caracterizacion[cols_preview]
    .round(4)
    .head(10)
)

### Resultados segmentación

#### Tablas de métricas de segmentación y relación con atributos de la masa

Esta celda resume los resultados obtenidos después de ejecutar el batch. Primero se muestran las métricas globales de la segmentación híbrida. Después se listan los casos con menor Dice para revisar visualmente. Finalmente, se calculan resúmenes agrupados según atributos del CSV, como forma de la masa, densidad mamaria, márgenes y patología.


In [ ]:
# @title
# ============================================================
# CELDA 11 - Resumen de métricas y relación con atributos del CSV
# ============================================================

# Esta celda usa df_caracterizacion generado en la celda anterior.
# No vuelve a ejecutar el pipeline ni recalcula segmentaciones.

if "df_caracterizacion" not in globals():
    df_caracterizacion = pd.read_csv(OUT_DIR / "resultados_caracterizados.csv")

# ------------------------------------------------------------
# 1. Resumen global de métricas
# ------------------------------------------------------------

metricas_principales = [
    "dice",
    "jaccard",
    "precision",
    "sensibilidad",
    "ratio_area"
]

metricas_principales = [
    m for m in metricas_principales
    if m in df_caracterizacion.columns
]

resumen_global_metricas = (
    df_caracterizacion[metricas_principales]
    .agg(["mean", "std", "median", "min", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "metrica"})
    .round(4)
)

print("Resumen global de métricas de la segmentación híbrida:")
display(resumen_global_metricas)

resumen_global_metricas.to_csv(
    OUT_DIR / "resumen_global_metricas_hibrido.csv",
    index=False
)

# ------------------------------------------------------------
# 2. Comparación cuantitativa entre métodos
# ------------------------------------------------------------

if "df_metodos" not in globals():
    ruta_metodos = OUT_DIR / "resultados_metodos_pipeline_final.csv"

    if ruta_metodos.exists():
        df_metodos = pd.read_csv(ruta_metodos)
    else:
        raise FileNotFoundError(
            "No existe df_metodos ni el archivo resultados_metodos_pipeline_final.csv. "
            "Primero corré la CELDA 9."
        )

df_metodos_comp = df_metodos.copy()

df_metodos_comp["metodo"] = df_metodos_comp["metodo"].replace({
    "GAC": "MorphGAC",
    "Kmeans": "K-means",
    "Hibrido": "Híbrido"
})

orden_metodos = ["MorphGAC", "K-means", "Híbrido"]

resumen_comparativo_metodos = (
    df_metodos_comp
    .groupby("metodo")
    .agg(
        n=("dice", "count"),
        dice_promedio=("dice", "mean"),
        jaccard_promedio=("jaccard", "mean"),
        precision_promedio=("precision", "mean"),
        sensibilidad_promedio=("sensibilidad", "mean"),
        ratio_area_promedio=("ratio_area", "mean")
    )
    .reindex(orden_metodos)
    .reset_index()
    .round(4)
)

print("\nComparación cuantitativa entre métodos de segmentación:")
display(resumen_comparativo_metodos)

resumen_comparativo_metodos.to_csv(
    OUT_DIR / "resumen_comparativo_metodos.csv",
    index=False
)

# ------------------------------------------------------------
# 3. Casos a revisar
# ------------------------------------------------------------

print("\nCantidad de casos marcados para revisión visual:")

if "revisar_segmentacion" in df_caracterizacion.columns:
    display(
        df_caracterizacion["revisar_segmentacion"]
        .value_counts(dropna=False)
        .rename_axis("revisar_segmentacion")
        .reset_index(name="cantidad")
    )

cols_peores = [
    "case_id",
    "patient_id",
    "lateralidad",
    "vista",
    "mass_shape",
    "mass_margins",
    "breast_density",
    "pathology",
    "dice",
    "jaccard",
    "precision",
    "sensibilidad",
    "ratio_area",
    "diagnostico_area",
    "gt_area_px",
    "seg_area_px",
    "delta_area_px",
    "gt_circularidad",
    "seg_circularidad",
    "delta_circularidad",
    "gt_irregularidad_borde",
    "seg_irregularidad_borde",
    "delta_irregularidad_borde",
    "gt_solidez",
    "seg_solidez",
    "delta_solidez",
    "revisar_segmentacion"
]

cols_peores = [
    c for c in cols_peores
    if c in df_caracterizacion.columns
]

peores_dice = (
    df_caracterizacion
    .sort_values("dice")
    [cols_peores]
    .head(10)
    .round(4)
)

print("\nCasos con menor Dice:")
display(peores_dice)

peores_dice.to_csv(
    OUT_DIR / "peores_casos_por_dice.csv",
    index=False
)

# ------------------------------------------------------------
# 3. Función para resumir por categorías del CSV
# ------------------------------------------------------------

def resumen_por_categoria(df, columna):
    """
    Resume métricas de segmentación agrupando por una columna categórica.
    """

    if columna not in df.columns:
        return pd.DataFrame()

    resumen = (
        df
        .groupby(columna, dropna=False)
        .agg(
            n=("dice", "count"),
            dice_promedio=("dice", "mean"),
            dice_mediana=("dice", "median"),
            jaccard_promedio=("jaccard", "mean"),
            precision_promedio=("precision", "mean"),
            sensibilidad_promedio=("sensibilidad", "mean"),
            ratio_area_promedio=("ratio_area", "mean"),
            n_revisar=("revisar_segmentacion", "sum")
        )
        .reset_index()
    )

    resumen["porcentaje_revisar"] = (
        100 * resumen["n_revisar"] / resumen["n"]
    )

    return resumen.round(4)

# ------------------------------------------------------------
# 5. Tablas por atributos del CSV
# ------------------------------------------------------------

tabla_shape = resumen_por_categoria(df_caracterizacion, "mass_shape")
tabla_margins = resumen_por_categoria(df_caracterizacion, "mass_margins")
tabla_density = resumen_por_categoria(df_caracterizacion, "breast_density")
tabla_pathology = resumen_por_categoria(df_caracterizacion, "pathology")

# Guardado de tablas
tabla_shape.to_csv(OUT_DIR / "tabla_por_mass_shape.csv", index=False)
tabla_margins.to_csv(OUT_DIR / "tabla_por_mass_margins.csv", index=False)
tabla_density.to_csv(OUT_DIR / "tabla_por_breast_density.csv", index=False)
tabla_pathology.to_csv(OUT_DIR / "tabla_por_pathology.csv", index=False)

print("\nResumen por forma de la masa:")
display(tabla_shape)

print("\nResumen por densidad mamaria:")
display(tabla_density)

print("\nResumen por márgenes:")
display(tabla_margins)

print("\nTabla adicional guardada en Drive:")
print(OUT_DIR / "tabla_por_pathology.csv")

#### Gráficos de resultados

Esta celda genera las figuras principales para resumir el desempeño del pipeline. Primero se resume el Dice de cada método mediante boxplots, para ver no sólo el promedio sino también la dispersión entre casos. Luego se comparan precisión y sensibilidad promedio.

Después se analiza el Dice del método híbrido según forma de la masa y densidad mamaria, usando los atributos del CSV.


In [ ]:
# @title
# ============================================================
# CELDA 12 - Gráficos finales de resultados
# ============================================================

OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Carga de resultados si se reabrió el notebook
# ------------------------------------------------------------

if "df_metodos" not in globals():
    ruta_metodos = OUT_DIR / "resultados_metodos_pipeline_final.csv"
    if ruta_metodos.exists():
        df_metodos = pd.read_csv(ruta_metodos)

if "df_caracterizacion" not in globals():
    ruta_caracterizacion = OUT_DIR / "resultados_caracterizados.csv"
    if ruta_caracterizacion.exists():
        df_caracterizacion = pd.read_csv(ruta_caracterizacion)

# ------------------------------------------------------------
# Resumen comparativo de métodos sobre ventanas preprocesadas
# ------------------------------------------------------------

if "df_metodos" in globals() and len(df_metodos) > 0:

    resumen_metodos = (
        df_metodos
        .groupby("metodo")
        .agg(
            n=("dice", "count"),
            dice_promedio=("dice", "mean"),
            dice_mediana=("dice", "median"),
            jaccard_promedio=("jaccard", "mean"),
            precision_promedio=("precision", "mean"),
            sensibilidad_promedio=("sensibilidad", "mean"),
            ratio_area_promedio=("ratio_area", "mean")
        )
        .reset_index()
        .sort_values("dice_promedio", ascending=False)
        .round(4)
    )

resumen_metodos.to_csv(
    OUT_DIR / "resumen_comparativo_metodos.csv",
    index=False)

print("Resumen comparativo de métodos guardado en:")
print(OUT_DIR / "resumen_comparativo_metodos.csv")


# ------------------------------------------------------------
# Función auxiliar: boxplot con puntos individuales
# ------------------------------------------------------------

def boxplot_con_puntos(
    df,
    columna_categoria,
    columna_valor,
    orden,
    colores,
    titulo,
    xlabel,
    ylabel,
    nombre_archivo
):
    datos = []
    etiquetas = []
    colores_usados = []
    rng = np.random.default_rng(0)

    for categoria in orden:
        serie = df[df[columna_categoria] == categoria][columna_valor].dropna()
        datos.append(serie.values)
        etiquetas.append(f"{categoria}\nn={len(serie)}")
        colores_usados.append(colores.get(categoria, "#B0B0B0"))

    fig, ax = plt.subplots(figsize=(max(7, 1.6 * len(orden)), 5))

    bp = ax.boxplot(
        datos,
        patch_artist=True,
        showmeans=True,
        meanprops=dict(
            marker="o",
            markerfacecolor="white",
            markeredgecolor="#333333",
            markersize=5
        ),
        medianprops=dict(color="#333333", linewidth=1.5),
        whiskerprops=dict(color="#666666"),
        capprops=dict(color="#666666"),
        flierprops=dict(
            marker="o",
            markersize=3,
            markerfacecolor="#999999",
            markeredgecolor="#999999",
            alpha=0.5
        )
    )

    for i, (caja, color, valores) in enumerate(zip(bp["boxes"], colores_usados, datos), start=1):
        caja.set_facecolor(color)
        caja.set_alpha(0.85)

        if len(valores) > 0:
            jitter = rng.normal(i, 0.045, size=len(valores))
            ax.scatter(
                jitter,
                valores,
                s=18,
                color="#4D4D4D",
                alpha=0.55,
                zorder=3
            )

    ax.set_xticklabels(etiquetas)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(titulo)
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig(FIG_DIR / nombre_archivo, dpi=150, bbox_inches="tight")
    plt.show()


# ------------------------------------------------------------
# 1) Boxplot de Dice: GAC vs K-means vs híbrido
# ------------------------------------------------------------

if "df_metodos" in globals() and len(df_metodos) > 0:

    df_metodos_plot = df_metodos.copy()

    df_metodos_plot["metodo_plot"] = df_metodos_plot["metodo"].replace({
        "GAC": "GAC",
        "Kmeans": "K-means",
        "Hibrido": "Híbrido"
    })

    orden_metodos = ["GAC", "K-means", "Híbrido"]

    colores_metodos = {
        "GAC": "#8FBBD9",
        "K-means": "#A8C686",
        "Híbrido": "#B8A9C9"
    }

    boxplot_con_puntos(
        df_metodos_plot,
        "metodo_plot",
        "dice",
        orden_metodos,
        colores_metodos,
        "Distribución del índice de Dice por método",
        "Método de segmentación",
        "Índice de Dice",
        "boxplot_dice_por_metodo.png"
    )


# ------------------------------------------------------------
# 2) Precisión y sensibilidad promedio por método
# ------------------------------------------------------------

if "df_metodos_plot" in globals() and len(df_metodos_plot) > 0:

    resumen_ps = (
        df_metodos_plot
        .groupby("metodo_plot")
        .agg(
            precision_media=("precision", "mean"),
            precision_std=("precision", "std"),
            sensibilidad_media=("sensibilidad", "mean"),
            sensibilidad_std=("sensibilidad", "std")
        )
        .reindex(orden_metodos)
    )

    x = np.arange(len(orden_metodos))
    width = 0.34

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.bar(
        x - width / 2,
        resumen_ps["precision_media"],
        width,
        yerr=resumen_ps["precision_std"],
        capsize=4,
        label="Precisión",
        color="#8DD3C7",
        alpha=0.9
    )

    ax.bar(
        x + width / 2,
        resumen_ps["sensibilidad_media"],
        width,
        yerr=resumen_ps["sensibilidad_std"],
        capsize=4,
        label="Sensibilidad",
        color="#FDBE85",
        alpha=0.9
    )

    ax.set_xticks(x)
    ax.set_xticklabels(orden_metodos)
    ax.set_ylabel("Valor medio ± desvío estándar")
    ax.set_title("Equilibrio entre precisión y sensibilidad")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

    plt.tight_layout()
    plt.savefig(FIG_DIR / "precision_sensibilidad_por_metodo.png", dpi=150, bbox_inches="tight")
    plt.show()


# ------------------------------------------------------------
# 3) Boxplot de Dice según forma de masa
# ------------------------------------------------------------

if "df_caracterizacion" in globals() and "mass_shape" in df_caracterizacion.columns:

    orden_shape = [
        x for x in ["ROUND", "OVAL", "LOBULATED", "IRREGULAR"]
        if x in df_caracterizacion["mass_shape"].dropna().unique()
    ]

    colores_shape = {
        "ROUND": "#8DD3C7",
        "OVAL": "#F6E8A8",
        "LOBULATED": "#B8A9C9",
        "IRREGULAR": "#D8A7A7"
    }

    if len(orden_shape) > 0:
        boxplot_con_puntos(
            df_caracterizacion,
            "mass_shape",
            "dice",
            orden_shape,
            colores_shape,
            "Índice de Dice según forma de la masa",
            "Forma de la masa",
            "Índice de Dice",
            "boxplot_dice_por_forma.png"
        )


# ------------------------------------------------------------
# 4) Boxplot de Dice según densidad mamaria
# ------------------------------------------------------------

if "df_caracterizacion" in globals() and "breast_density" in df_caracterizacion.columns:

    orden_density = sorted(df_caracterizacion["breast_density"].dropna().unique().tolist())

    colores_density = {
        d: color for d, color in zip(
            orden_density,
            ["#A8C686", "#8FBBD9", "#FDBE85", "#D8A7A7"]
        )
    }

    if len(orden_density) > 0:
        boxplot_con_puntos(
            df_caracterizacion,
            "breast_density",
            "dice",
            orden_density,
            colores_density,
            "Índice de Dice según densidad mamaria",
            "Densidad mamaria",
            "Índice de Dice",
            "boxplot_dice_por_densidad.png"
        )


# ------------------------------------------------------------
# 5) Precisión vs sensibilidad del método híbrido
# ------------------------------------------------------------

if "df_caracterizacion" in globals():

    fig, ax = plt.subplots(figsize=(6, 5))

    for forma, grupo in df_caracterizacion.groupby("mass_shape"):
        ax.scatter(
            grupo["sensibilidad"],
            grupo["precision"],
            s=45,
            alpha=0.75,
            label=f"{forma} (n={len(grupo)})"
        )

    ax.axhline(0.70, color="#999999", linestyle="--", linewidth=1)
    ax.axvline(0.70, color="#999999", linestyle="--", linewidth=1)

    ax.set_xlabel("Sensibilidad")
    ax.set_ylabel("Precisión")
    ax.set_title("Relación precisión-sensibilidad del método híbrido")
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc="lower left")

    plt.tight_layout()
    plt.savefig(FIG_DIR / "precision_vs_sensibilidad_hibrido.png", dpi=150, bbox_inches="tight")
    plt.show()


print("\nFiguras guardadas en:", FIG_DIR)


### Resultados de extracción de características

#### Tablas de extracción de características

Esta sección resume las características calculadas sobre la máscara de referencia y sobre la máscara segmentada por el método híbrido. Para cada característica se compara el valor obtenido en la máscara de referencia (`gt_`) con el valor obtenido en la máscara segmentada (`seg_`).


In [ ]:
# @title
# ============================================================
# CELDA 13 - Tabla resumen de características GT vs segmentación
# ============================================================

if "df_caracterizacion" not in globals():
    df_caracterizacion = pd.read_csv(OUT_DIR / "resultados_caracterizados.csv")

caracteristicas_seleccionadas = [
    "area_px",
    "perimetro_px",
    "circularidad",
    "irregularidad_borde",
    "solidez",
    "intensidad_media",
    "intensidad_std"
]

# Me quedo sólo con las características que existan en GT y segmentación.
caracteristicas_seleccionadas = [
    f for f in caracteristicas_seleccionadas
    if f"gt_{f}" in df_caracterizacion.columns
    and f"seg_{f}" in df_caracterizacion.columns
]

filas_resumen_car = []

for f in caracteristicas_seleccionadas:

    gt_col = f"gt_{f}"
    seg_col = f"seg_{f}"
    delta_col = f"delta_{f}"

    if delta_col not in df_caracterizacion.columns:
        df_caracterizacion[delta_col] = (
            df_caracterizacion[seg_col] - df_caracterizacion[gt_col]
        )

    gt = df_caracterizacion[gt_col]
    seg = df_caracterizacion[seg_col]
    delta = df_caracterizacion[delta_col]

    # Diferencia relativa: se calcula respecto del valor GT.
    # Se evita dividir por cero reemplazando ceros por NaN.
    delta_rel = 100 * delta / gt.replace(0, np.nan)

    filas_resumen_car.append({
        "caracteristica": f,
        "n": int(pd.concat([gt, seg], axis=1).dropna().shape[0]),
        "gt_media": gt.mean(),
        "gt_std": gt.std(),
        "seg_media": seg.mean(),
        "seg_std": seg.std(),
        "delta_media": delta.mean(),
        "delta_std": delta.std(),
        "delta_relativa_media_%": delta_rel.mean(),
        "delta_relativa_std_%": delta_rel.std()
    })

df_resumen_caracteristicas = pd.DataFrame(filas_resumen_car).round(4)

df_resumen_caracteristicas.to_csv(
    OUT_DIR / "resumen_caracteristicas_gt_vs_seg.csv",
    index=False
)

print("Resumen de características GT vs segmentación:")
display(df_resumen_caracteristicas)

print("\nTabla guardada en:")
print(OUT_DIR / "resumen_caracteristicas_gt_vs_seg.csv")

#### Gráficos de características

El primer gráfico compara los valores medios de algunas características entre la máscara de referencia y la máscara segmentada. Se muestran también los desvíos estándar para representar la variabilidad entre casos.

El segundo resume cuánto cambia cada característica en la segmentación respecto de la referencia. Para eso se usa el cociente segmentación / referencia. Un valor igual a 1 indica coincidencia promedio entre ambas, un valor mayor que 1 indica que la característica tiende a aumentar en la segmentación y un valor menor que 1 indica que tiende a disminuir.


In [ ]:
# @title
# ============================================================
# CELDA 13 - Resumen de comparación de características
# GT vs segmentación
# ============================================================

# Si no está df_caracterizacion en memoria, intento cargarlo desde Drive.
if "df_caracterizacion" not in globals():
    ruta_caracterizacion = OUT_DIR / "resultados_caracterizados.csv"

    if ruta_caracterizacion.exists():
        df_caracterizacion = pd.read_csv(ruta_caracterizacion)
    else:
        raise FileNotFoundError(
            "No existe df_caracterizacion ni el archivo resultados_caracterizados.csv. "
            "Primero corré la CELDA 10."
        )

features_car = [
    "area_px",
    "perimetro_px",
    "circularidad",
    "irregularidad_borde",
    "solidez",
    "intensidad_media",
    "intensidad_std"
]

# Me quedo sólo con las características que existan para GT y segmentación.
features_car = [
    f for f in features_car
    if f"gt_{f}" in df_caracterizacion.columns
    and f"seg_{f}" in df_caracterizacion.columns
]

filas_resumen = []

for f in features_car:
    gt = df_caracterizacion[f"gt_{f}"].astype(float)
    seg = df_caracterizacion[f"seg_{f}"].astype(float)

    delta = seg - gt
    ratio = seg / gt.replace(0, np.nan)

    filas_resumen.append({
        "caracteristica": f,
        "gt_media": gt.mean(),
        "gt_std": gt.std(),
        "seg_media": seg.mean(),
        "seg_std": seg.std(),
        "delta_media_seg_menos_gt": delta.mean(),
        "delta_std_seg_menos_gt": delta.std(),
        "ratio_medio_seg_sobre_gt": ratio.mean(),
        "ratio_std_seg_sobre_gt": ratio.std()
    })

resumen_comparacion = pd.DataFrame(filas_resumen).round(4)

resumen_comparacion.to_csv(
    OUT_DIR / "resumen_comparacion_caracteristicas.csv",
    index=False
)

print("Resumen de comparación de características:")
display(resumen_comparacion)

print("\nTabla guardada en:")
print(OUT_DIR / "resumen_comparacion_caracteristicas.csv")

In [ ]:
# @title
# ============================================================
# CARACTERÍSTICAS - Gráfico para informe
# Barras referencia vs segmentación con desvío estándar
# ============================================================

if "resumen_comparacion" not in globals():
    resumen_comparacion = pd.read_csv(OUT_DIR / "resumen_comparacion_caracteristicas.csv")

features_bar = [
    "circularidad",
    "irregularidad_borde",
    "solidez",
    "intensidad_media",
    "intensidad_std"
]

features_bar = [
    f for f in features_bar
    if f in resumen_comparacion["caracteristica"].values
]

labels_bar = {
    "circularidad": "Circularidad",
    "irregularidad_borde": "Irregularidad\ndel borde",
    "solidez": "Solidez",
    "intensidad_media": "Intensidad\nmedia",
    "intensidad_std": "Desvío estándar\nde intensidad"
}

tabla_bar = (
    resumen_comparacion
    .set_index("caracteristica")
    .loc[features_bar]
    .reset_index()
)

medias_gt = tabla_bar["gt_media"].values
std_gt = tabla_bar["gt_std"].values

medias_seg = tabla_bar["seg_media"].values
std_seg = tabla_bar["seg_std"].values

x = np.arange(len(features_bar))
width = 0.34

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    x - width / 2,
    medias_gt,
    width,
    yerr=std_gt,
    capsize=4,
    label="Referencia",
    color="#80B1D3",
    alpha=0.9
)

ax.bar(
    x + width / 2,
    medias_seg,
    width,
    yerr=std_seg,
    capsize=4,
    label="Segmentación",
    color="#BC80BD",
    alpha=0.9
)

ax.set_xticks(x)
ax.set_xticklabels([labels_bar[f] for f in features_bar])
ax.set_ylabel("Valor medio ± desvío estándar")
ax.set_title("Características de las máscaras: referencia vs segmentación")
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()

plt.savefig(
    FIG_DIR / "caracteristicas_referencia_vs_segmentacion_informe.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# @title
# ============================================================
# CARACTERÍSTICAS - Ratio segmentación / referencia
# Para informe o anexo
# ============================================================

if "resumen_comparacion" not in globals():
    ruta_resumen_car = OUT_DIR / "resumen_comparacion_caracteristicas.csv"

    if ruta_resumen_car.exists():
        resumen_comparacion = pd.read_csv(ruta_resumen_car)
    else:
        raise FileNotFoundError(
            "No existe resumen_comparacion ni el archivo resumen_comparacion_caracteristicas.csv. "
            "Primero corré la celda de resumen de comparación de características."
        )

features_ratio = [
    "area_px",
    "perimetro_px",
    "circularidad",
    "irregularidad_borde",
    "solidez",
    "intensidad_media",
    "intensidad_std"
]

features_ratio = [
    f for f in features_ratio
    if f in resumen_comparacion["caracteristica"].values
]

labels_ratio = {
    "area_px": "Área",
    "perimetro_px": "Perímetro",
    "circularidad": "Circularidad",
    "irregularidad_borde": "Irregularidad\ndel borde",
    "solidez": "Solidez",
    "intensidad_media": "Intensidad\nmedia",
    "intensidad_std": "Desvío estándar\nde intensidad"
}

# Reordeno la tabla según el orden definido en features_ratio
tabla_ratio = (
    resumen_comparacion
    .set_index("caracteristica")
    .loc[features_ratio]
    .reset_index()
)

x = np.arange(len(tabla_ratio))

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(
    x,
    tabla_ratio["ratio_medio_seg_sobre_gt"],
    yerr=tabla_ratio["ratio_std_seg_sobre_gt"],
    capsize=4,
    color="#8DD3C7",
    alpha=0.9,
    edgecolor="#5F9F96",
    linewidth=0.8
)

ax.axhline(
    1,
    color="#666666",
    linestyle="--",
    linewidth=1.2,
    label="Referencia = segmentación"
)

ax.set_xticks(x)
ax.set_xticklabels(
    [labels_ratio[f] for f in tabla_ratio["caracteristica"]],
    rotation=35,
    ha="right"
)

ax.set_ylabel("Ratio medio segmentación / referencia")
ax.set_title("Cambio relativo de las características segmentadas")
ax.grid(axis="y", alpha=0.25)
ax.legend()

plt.tight_layout()

plt.savefig(
    FIG_DIR / "ratio_caracteristicas_segmentacion_referencia_informe.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()